<font color = Lime>

# Detailed plot tool

## Initial grouping code

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)   # 30 columns

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    # Examples:
    # Cyl100_Re100k_Clean
    # Cyl100_Re100k_Trip_0.15_A57

    meta = {
        "cylinder": parts[0],
        "re": parts[1],          # e.g. Re100k
        "case": None,            # Clean or Trip
        "trip_mm": None,         # e.g. 0.15 or 0.94
        "angle": None            # e.g. 57
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = parts[3]
        meta["angle"] = parts[4].replace("A", "")

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

# Grouped storage
grouped_results = defaultdict(list)

for file in DATA_FOLDER.glob("*.lvm"):
    FILE_PATH = file
    meta = parse_filename(FILE_PATH)

    df = read_lvm_file(FILE_PATH)

    # -----------------------------
    # Extract the two pressure layers
    # -----------------------------
    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)   # shape: (n_samples, 30)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)   # shape: (n_samples, 30)

    theta1 = np.arange(0.0, 360.0, 12.0)
    theta2 = theta1 + 6.0
    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    # -----------------------------
    # Convert pressure to Cp
    # -----------------------------
    # Using mean dynamic pressure keeps consistency with your existing Cp definition
    q_inf_mean = df.iloc[:, 63].mean()   # 64th column = pitot channel

    # Time-resolved Cp at each tap
    cp1_samples = layer1 / q_inf_mean    # shape: (n_samples, 30)
    cp2_samples = layer2 / q_inf_mean    # shape: (n_samples, 30)

    # Mean Cp at each tap
    cp1_mean = cp1_samples.mean(axis=0)
    cp2_mean = cp2_samples.mean(axis=0)

    # Variance of Cp fluctuations at each tap
    cp1_var = cp1_samples.var(axis=0)
    cp2_var = cp2_samples.var(axis=0)

    # RMS of Cp fluctuations at each tap
    # This is the same as standard deviation of Cp
    cp1_rms = np.sqrt(cp1_var)
    cp2_rms = np.sqrt(cp2_var)

    # -----------------------------
    # Collapse onto one 6-degree grid
    # -----------------------------
    cp_collapsed = np.empty(theta_collapsed.size)
    cp_var_collapsed = np.empty(theta_collapsed.size)
    cp_rms_collapsed = np.empty(theta_collapsed.size)

    # keep your working ordering
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    cp_var_collapsed[0::2] = cp2_var
    cp_var_collapsed[1::2] = cp1_var

    cp_rms_collapsed[0::2] = cp2_rms
    cp_rms_collapsed[1::2] = cp1_rms

    # repeat first point at end to close the loop
    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    cp_collapsed_closed = np.r_[cp_collapsed, cp_collapsed[0]]
    cp_var_collapsed_closed = np.r_[cp_var_collapsed, cp_var_collapsed[0]]
    cp_rms_collapsed_closed = np.r_[cp_rms_collapsed, cp_rms_collapsed[0]]

    # -----------------------------
    # Group the results
    # -----------------------------
    if meta["case"] == "Clean":
        group_key = ("Clean",)
    else:
        group_key = ("Trip", meta["trip_mm"], meta["angle"])

    grouped_results[group_key].append({
        "file_name": FILE_PATH.name,
        "re": meta["re"],
        "theta": theta_collapsed_closed,
        "cp": cp_collapsed_closed,
        "cp_var": cp_var_collapsed_closed,
        "cp_rms": cp_rms_collapsed_closed
    })


print(q_inf_mean)



In [ ]:
q_inf_mean = df.iloc[:, 63].describe()
print(q_inf_mean)

# Notes (For instance color coding to make navigation easier)

## Color codes

<font color = Red>

If Red, it needs adressing


<font color = Yellow>

If Yellow, its an initial draft, that works, but acccuracy and correctness is still to be confirmed. 

# C_P related plots

## Specific angle and tripwire plotter, combined Reynolds number

### This code prints out the combined graph for a specific tripwire angle and thickness. All reynold numbers are combined into the same graph, to give an idea of changes due to reynolds number.

In [ ]:
# Example 1:
# Plot all runs for 0.94 mm wire at angle 57


target_key = ("Trip", "0.94", "63")

if target_key in grouped_results:
    plt.figure(figsize=(9, 6))

    # sort by Reynolds number numerically
    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(grouped_results[target_key], key=re_value):
        plt.plot(
            run_data["theta"],
            run_data["cp"],
            "o-",
            label=run_data["re"]
        )
    # =========================
    # Tripwire location markers
    # =========================
    trip_angle = float(target_key[2])
    trip_angle_upper = trip_angle
    trip_angle_lower = 360 - trip_angle

    plt.axvline(
        trip_angle_upper,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label="Tripwire"
    )

    plt.axvline(
        trip_angle_lower,
        color="black",
        linestyle="--",
        linewidth=1.5
    )



    plt.xlabel("Angle [deg]")
    plt.ylabel("$C_p$")
    plt.ylim(-3.5, 1.5)
    plt.title(f"Collapsed $C_p$ for trip wire {target_key[1]} mm at A{target_key[2]}")
    plt.grid(True)
    plt.legend(title="Re")
    plt.show()

else:
    print(f"No files found for group {target_key}")

## Clean cylinder run

### Clean cylinder has its own since the name sorting of the code doesnt work the same for the clean files. In this code all reynolds numbers are combined into the same graph.

In [ ]:
if ("Clean",) in grouped_results:
    plt.figure(figsize=(9, 6))

    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(grouped_results[("Clean",)], key=re_value):
        plt.plot(
            run_data["theta"],
            run_data["cp"],
            "o-",
            label=run_data["re"]
        )
    plt.xlabel("Angle [deg]")
    plt.ylabel("$C_p$")
    plt.title("Smooth cylinder day 1")
    plt.ylim(-3.5, 1.5)
    plt.grid(True)
    plt.legend(title="Re")

## Individual reynolds numbers for a specific thickness and angle

### Using the target key, this code displays all vallues collected at a spefific reynolds number, again for a pre set wire thickness and angle. They are printed in order after starting from Re = 50.000

In [ ]:
target_key = ("Trip", "0.94", "69")

if target_key in grouped_results:

    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(grouped_results[target_key], key=re_value):
        plt.figure(figsize=(9, 6))

        plt.plot(
            run_data["theta"],
            run_data["cp"],
            "o-",
            label=run_data["re"],
            color = "crimson"
        )
                        # =========================
        # Tripwire location markers
        # =========================
        trip_angle = float(target_key[2])
        trip_angle_upper = trip_angle
        trip_angle_lower = 360 - trip_angle

        plt.axvline(
            trip_angle_upper,
            color="black",
            linestyle="--",
            linewidth=1.5,
            label="Tripwire"
        )

        plt.axvline(
            trip_angle_lower,
            color="black",
            linestyle="--",
            linewidth=1.5
        )


        plt.xlabel("Angle [deg]")
        plt.ylabel("$C_p$")
        plt.title(f'{run_data["file_name"]}')
        plt.grid(True)
        plt.ylim(-3, 1.2)
        plt.legend()

        plt.xlabel("Angle [deg]")
        plt.ylabel("$C_p$")
        plt.title(f'{run_data["file_name"]}')
        plt.grid(True)
        plt.ylim(-1, 1.2)
        plt.legend()


    plt.show()

else:
    print(f"No files found for group {target_key}")

# C_P around the cylinder plots

<font color = tomato>

## C_P at each tap visualization, dot color

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt


# target_key = ("Trip", "0.94", "57")
# plot_cp_on_cylinder(grouped_results, target_key)


# def re_value(run_dict):
#     return int(run_dict["re"].replace("Re", "").replace("k", ""))

# def plot_cp_on_cylinder(grouped_results, target_key, radius=1.0):
#     if target_key not in grouped_results:
#         print(f"No files found for group {target_key}")
#         return

#     runs = sorted(grouped_results[target_key], key=re_value)

#     # Common color scale across all plots in this group
#     cp_all = np.concatenate([run["cp"][:-1] for run in runs])   # drop repeated 360 point
#     vmin = cp_all.min()
#     vmax = cp_all.max()

#     for run_data in runs:
#         theta_deg = run_data["theta"][:-1]   # drop repeated closing point
#         cp_vals = run_data["cp"][:-1]

#         theta_rad = np.deg2rad(theta_deg)

#         # Front stagnation point at left side of the circle
#         x = -radius * np.cos(theta_rad)
#         y =  radius * np.sin(theta_rad)

#         fig, ax = plt.subplots(figsize=(7, 7))

#         # Cylinder outline
#         phi = np.linspace(0, 2*np.pi, 400)
#         ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.2)

#         # Colored pressure taps
#         sc = ax.scatter(
#             x, y,
#             c=cp_vals,
#             s=120,
#             cmap="coolwarm",
#             vmin=vmin,
#             vmax=vmax,
#             edgecolors="black",
#             linewidths=0.5
#         )

#         # Connect points in angular order
#         sort_idx = np.argsort(theta_deg)
#         ax.plot(x[sort_idx], y[sort_idx], color="gray", alpha=0.5, lw=1)

#         # Optional angle labels
#         for xd, yd, ang in zip(x, y, theta_deg):
#             ax.text(1.10*xd, 1.10*yd, f"{int(ang)}°", fontsize=7,
#                     ha="center", va="center")

#         cbar = plt.colorbar(sc, ax=ax)
#         cbar.set_label(r"$C_p$")

#         ax.set_aspect("equal")
#         ax.set_xlim(-1.25*radius, 1.25*radius)
#         ax.set_ylim(-1.25*radius, 1.25*radius)
#         ax.set_xlabel("x")
#         ax.set_ylabel("y")
#         ax.set_title(f'{run_data["file_name"]}')
#         ax.grid(True)

#         plt.show()

## C_P around the cylinder visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Example
# =========================
# target_key = ("Trip", "0.94", "57")
# target_re = "Re100k"     # set to None to plot all Reynolds numbers
target_key = ("Clean",)
target_re = "Re100k"

def re_value(run_dict):
    return int(run_dict["re"].replace("Re", "").replace("k", ""))

def plot_cp_normal_bars(grouped_results, target_key, target_re=None, radius=1.0, max_bar_length=0.35, cp_abs_max=None):
    if target_key not in grouped_results:
        print(f"No files found for group {target_key}")
        return

    runs = sorted(grouped_results[target_key], key=re_value)

    # Filter by Reynolds number if requested
    if target_re is not None:
        runs = [run for run in runs if run["re"] == target_re]

        if not runs:
            print(f"No files found for group {target_key} with Reynolds number {target_re}")
            return

# Use supplied common scale, or calculate local scale if none is given
    if cp_abs_max is None:
        cp_all = np.concatenate([run["cp"][:-1] for run in runs])
        cp_abs_max = np.max(np.abs(cp_all))

    if cp_abs_max == 0:
        cp_abs_max = 1.0

    scale = max_bar_length / cp_abs_max

    for run_data in runs:
        theta_deg = run_data["theta"][:-1]   # drop repeated 360° closing point
        cp_vals = run_data["cp"][:-1]
        theta_rad = np.deg2rad(theta_deg)

        # Cylinder surface coordinates
        x = -radius * np.cos(theta_rad)
        y =  radius * np.sin(theta_rad)

        # Outward unit normals
        nx = -np.cos(theta_rad)
        ny =  np.sin(theta_rad)

        fig, ax = plt.subplots(figsize=(7, 7))

        # Cylinder outline
        phi = np.linspace(0, 2*np.pi, 400)
        ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.5,)

        # Draw Cp bars normal to the surface
        for xs, ys, nxi, nyi, cp in zip(x, y, nx, ny, cp_vals):
            bar_len = scale * cp
            xe = xs + bar_len * nxi
            ye = ys + bar_len * nyi

            color = "crimson" if cp >= 0 else "royalblue"
            ax.plot([xs, xe], [ys, ye], color=color, lw=2.5, solid_capstyle="round")
            ax.plot(xs, ys, "ko", ms=3)

        # Angle labels every 30 degrees
        for xs, ys, ang in zip(x, y, theta_deg):
            if ang % 30 == 0:
                ax.text(
                    1.12 * xs,
                    1.12 * ys,
                    f"{int(ang)}°",
                    fontsize=8,
                    ha="center",
                    va="center"
                )

        ax.set_aspect("equal")
        ax.set_xlim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
        ax.set_ylim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title(f'{run_data["file_name"]}\nNormal-bar view of $C_p$')
        ax.grid(True)
        plt.show()

COMMON_CP_ABS_MAX = 2

plot_cp_normal_bars(grouped_results, target_key, target_re=target_re, cp_abs_max=COMMON_CP_ABS_MAX)

## C_P around cylinder with tripwire marker

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

target_key = ("Trip", "0.94", "57")
target_re = "Re100k"     # set to None to plot all Reynolds numbers
# target_key = ("Clean",)
# target_re = "Re150k"

def re_value(run_dict):
    return int(run_dict["re"].replace("Re", "").replace("k", ""))

def get_trip_angles(run_data, target_key):
    """
    Return trip-wire angles in degrees for plotting.
    Assumes trip wires are placed symmetrically on upper/lower side.
    """
    if target_key[0] != "Trip":
        return []

    if "angle" in run_data:
        trip_angle = float(run_data["angle"])
    else:
        trip_angle = float(target_key[-1])

    return [trip_angle, 360.0 - trip_angle]

def draw_trip_wire_arc(ax, center_angle_deg, radius=1.0,
                       arc_halfwidth_deg=2.5,
                       arc_radius_scale=1.015,
                       color="darkorange",
                       lw=4,
                       show_center_dot=True):
    """
    Draw a short arc to mark the trip-wire location.
    The arc is slightly outside the cylinder radius so it stays visible.
    """
    arc_angles_deg = np.linspace(
        center_angle_deg - arc_halfwidth_deg,
        center_angle_deg + arc_halfwidth_deg,
        80
    )
    arc_angles_rad = np.deg2rad(arc_angles_deg)

    r_arc = radius * arc_radius_scale

    x_arc = -r_arc * np.cos(arc_angles_rad)
    y_arc =  r_arc * np.sin(arc_angles_rad)

    ax.plot(
        x_arc, y_arc,
        color=color,
        lw=lw,
        solid_capstyle="round",
        zorder=6
    )

    if show_center_dot:
        ang_rad = np.deg2rad(center_angle_deg)
        xt = -r_arc * np.cos(ang_rad)
        yt =  r_arc * np.sin(ang_rad)

        ax.plot(
            xt, yt,
            marker="o",
            ms=6,
            mec="black",
            mfc="gold",
            zorder=7
        )

def plot_cp_normal_bars(grouped_results, target_key, target_re=None,
                        radius=1.0, max_bar_length=0.35,
                        show_trip_arc=True,
                        trip_arc_halfwidth_deg=2.5):
    if target_key not in grouped_results:
        print(f"No files found for group {target_key}")
        return

    runs = sorted(grouped_results[target_key], key=re_value)

    # Filter by Reynolds number if requested
    if target_re is not None:
        runs = [run for run in runs if run["re"] == target_re]

        if not runs:
            print(f"No files found for group {target_key} with Reynolds number {target_re}")
            return

    # Use one common scale for the selected runs
    cp_all = np.concatenate([run["cp"][:-1] for run in runs])
    cp_abs_max = np.max(np.abs(cp_all))

    if cp_abs_max == 0:
        cp_abs_max = 1.0

    scale = max_bar_length / cp_abs_max

    for run_data in runs:
        theta_deg = run_data["theta"][:-1]   # drop repeated 360° closing point
        cp_vals = run_data["cp"][:-1]
        theta_rad = np.deg2rad(theta_deg)

        # Cylinder surface coordinates
        x = -radius * np.cos(theta_rad)
        y =  radius * np.sin(theta_rad)

        # Outward unit normals
        nx = -np.cos(theta_rad)
        ny =  np.sin(theta_rad)

        fig, ax = plt.subplots(figsize=(7, 7))

        # Cylinder outline
        phi = np.linspace(0, 2*np.pi, 400)
        ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.5)

        # Draw Cp bars normal to the surface
        for xs, ys, nxi, nyi, cp in zip(x, y, nx, ny, cp_vals):
            bar_len = scale * cp
            xe = xs + bar_len * nxi
            ye = ys + bar_len * nyi

            color = "crimson" if cp >= 0 else "royalblue"
            ax.plot([xs, xe], [ys, ye], color=color, lw=2.5, solid_capstyle="round")
            ax.plot(xs, ys, "ko", ms=3)

        # -------------------------
        # Plot trip-wire arcs
        # -------------------------
        trip_angles = get_trip_angles(run_data, target_key)

        if show_trip_arc and trip_angles:
            for ang in trip_angles:
                draw_trip_wire_arc(
                    ax,
                    center_angle_deg=ang,
                    radius=radius,
                    arc_halfwidth_deg=trip_arc_halfwidth_deg,
                    arc_radius_scale=1.015,
                    color="darkorange",
                    lw=4,
                    show_center_dot=True
                )

        # Angle labels every 30 degrees
        for xs, ys, ang in zip(x, y, theta_deg):
            if ang % 30 == 0:
                ax.text(
                    1.12 * xs,
                    1.12 * ys,
                    f"{int(ang)}°",
                    fontsize=8,
                    ha="center",
                    va="center"
                )

        ax.set_aspect("equal")
        ax.set_xlim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
        ax.set_ylim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title(f'{run_data["file_name"]}\nNormal-bar view of $C_p$')
        ax.grid(True)

        plt.show()

plot_cp_normal_bars(
    grouped_results,
    target_key,
    target_re=target_re,
    show_trip_arc=True,
    trip_arc_halfwidth_deg=0.5
)

## C_P around the cylinder comparison code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Example
# =========================
target_key_1 = ("Trip", "0.94", "27")
target_re_1 = "Re250k"

target_key_2 = ("Trip", "0.94", "63")
target_re_2 = "Re250k"

def re_value(run_dict):
    return int(run_dict["re"].replace("Re", "").replace("k", ""))

def draw_cp_normal_bars(ax, run_data, cp_abs_max, radius=1.0, max_bar_length=0.35):
    theta_deg = run_data["theta"][:-1]   # drop repeated 360° point
    cp_vals = run_data["cp"][:-1]
    theta_rad = np.deg2rad(theta_deg)

    scale = max_bar_length / cp_abs_max if cp_abs_max != 0 else 1.0

    # Cylinder surface coordinates
    x = -radius * np.cos(theta_rad)
    y =  radius * np.sin(theta_rad)

    # Outward unit normals
    nx = -np.cos(theta_rad)
    ny =  np.sin(theta_rad)

    # Cylinder outline
    phi = np.linspace(0, 2*np.pi, 400)
    ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.5)

    # Draw Cp bars
    for xs, ys, nxi, nyi, cp in zip(x, y, nx, ny, cp_vals):
        bar_len = scale * cp
        xe = xs + bar_len * nxi
        ye = ys + bar_len * nyi

        color = "crimson" if cp >= 0 else "royalblue"
        ax.plot([xs, xe], [ys, ye], color=color, lw=2.5, solid_capstyle="round")
        ax.plot(xs, ys, "ko", ms=3)

    # Angle labels every 30 degrees
    for xs, ys, ang in zip(x, y, theta_deg):
        if ang % 30 == 0:
            ax.text(
                1.12 * xs,
                1.12 * ys,
                f"{int(ang)}°",
                fontsize=8,
                ha="center",
                va="center"
            )

    ax.set_aspect("equal")
    ax.set_xlim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_ylim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True)
    ax.set_title(f'{run_data["file_name"]}')

def get_filtered_run(grouped_results, target_key, target_re):
    if target_key not in grouped_results:
        raise ValueError(f"No files found for group {target_key}")

    runs = sorted(grouped_results[target_key], key=re_value)

    if target_re is not None:
        runs = [run for run in runs if run["re"] == target_re]

    if not runs:
        raise ValueError(f"No files found for group {target_key} with Reynolds number {target_re}")

    if len(runs) > 1:
        raise ValueError(
            f"More than one run matched {target_key} and {target_re}. "
            f"Expected exactly one."
        )

    return runs[0]

# Get the two selected runs
run1 = get_filtered_run(grouped_results, target_key_1, target_re_1)
run2 = get_filtered_run(grouped_results, target_key_2, target_re_2)

# Use common Cp scale across both plots
cp_abs_max = max(
    np.max(np.abs(run1["cp"][:-1])),
    np.max(np.abs(run2["cp"][:-1]))
)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

draw_cp_normal_bars(axes[0], run1, cp_abs_max)
draw_cp_normal_bars(axes[1], run2, cp_abs_max)

axes[0].set_title(f'{run1["file_name"]}')
axes[1].set_title(f'{run2["file_name"]}')

plt.tight_layout()
plt.show()



## C_P around the cylinder comparison, with tripwire

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Example
# =========================
target_key_1 = ("Trip", "0.94", "27")
target_re_1 = "Re250k"

target_key_2 = ("Trip", "0.94", "63")
target_re_2 = "Re250k"

def re_value(run_dict):
    return int(run_dict["re"].replace("Re", "").replace("k", ""))

def get_trip_angles_from_key(target_key):
    """
    Read trip-wire angles directly from target_key.
    Expected format for trip case: ("Trip", wire_diameter, angle)
    """
    if target_key[0] != "Trip":
        return []

    trip_angle = float(target_key[2])
    return [trip_angle, 360.0 - trip_angle]

def draw_trip_wire_arc(ax, center_angle_deg, radius=1.0,
                       arc_halfwidth_deg=2.5,
                       arc_radius_scale=1.015,
                       color="darkorange",
                       lw=4,
                       show_center_dot=True):
    arc_angles_deg = np.linspace(
        center_angle_deg - arc_halfwidth_deg,
        center_angle_deg + arc_halfwidth_deg,
        80
    )
    arc_angles_rad = np.deg2rad(arc_angles_deg)

    r_arc = radius * arc_radius_scale

    x_arc = -r_arc * np.cos(arc_angles_rad)
    y_arc =  r_arc * np.sin(arc_angles_rad)

    ax.plot(
        x_arc, y_arc,
        color=color,
        lw=lw,
        solid_capstyle="round",
        zorder=6
    )

    if show_center_dot:
        ang_rad = np.deg2rad(center_angle_deg)
        xt = -r_arc * np.cos(ang_rad)
        yt =  r_arc * np.sin(ang_rad)

        ax.plot(
            xt, yt,
            marker="o",
            ms=6,
            mec="black",
            mfc="gold",
            zorder=7
        )

def draw_cp_normal_bars(ax, run_data, cp_abs_max, trip_angles=None,
                        radius=1.0, max_bar_length=0.35,
                        show_trip_arc=True, trip_arc_halfwidth_deg=0.5):
    theta_deg = np.array(run_data["theta"][:-1])   # drop repeated 360° point
    cp_vals = np.array(run_data["cp"][:-1])
    theta_rad = np.deg2rad(theta_deg)

    scale = max_bar_length / cp_abs_max if cp_abs_max != 0 else 1.0

    # Cylinder surface coordinates
    x = -radius * np.cos(theta_rad)
    y =  radius * np.sin(theta_rad)

    # Outward unit normals
    nx = -np.cos(theta_rad)
    ny =  np.sin(theta_rad)

    # Cylinder outline
    phi = np.linspace(0, 2*np.pi, 400)
    ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.5)

    # Draw Cp bars
    for xs, ys, nxi, nyi, cp in zip(x, y, nx, ny, cp_vals):
        bar_len = scale * cp
        xe = xs + bar_len * nxi
        ye = ys + bar_len * nyi

        color = "crimson" if cp >= 0 else "royalblue"
        ax.plot([xs, xe], [ys, ye], color=color, lw=2.5, solid_capstyle="round")
        ax.plot(xs, ys, "ko", ms=3)

    # Plot trip-wire arcs
    if show_trip_arc and trip_angles:
        for ang in trip_angles:
            draw_trip_wire_arc(
                ax,
                center_angle_deg=ang,
                radius=radius,
                arc_halfwidth_deg=trip_arc_halfwidth_deg,
                arc_radius_scale=1.015,
                color="darkorange",
                lw=4,
                show_center_dot=True
            )

    # Angle labels every 30 degrees
    for xs, ys, ang in zip(x, y, theta_deg):
        if ang % 30 == 0:
            ax.text(
                1.12 * xs,
                1.12 * ys,
                f"{int(ang)}°",
                fontsize=8,
                ha="center",
                va="center"
            )

    ax.set_aspect("equal")
    ax.set_xlim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_ylim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True)
    ax.set_title(f'{run_data["file_name"]}')

def get_filtered_run(grouped_results, target_key, target_re):
    if target_key not in grouped_results:
        raise ValueError(f"No files found for group {target_key}")

    runs = sorted(grouped_results[target_key], key=re_value)

    if target_re is not None:
        runs = [run for run in runs if run["re"] == target_re]

    if not runs:
        raise ValueError(f"No files found for group {target_key} with Reynolds number {target_re}")

    if len(runs) > 1:
        raise ValueError(
            f"More than one run matched {target_key} and {target_re}. "
            f"Expected exactly one."
        )

    return runs[0]


# Get the two selected runs
run1 = get_filtered_run(grouped_results, target_key_1, target_re_1)
run2 = get_filtered_run(grouped_results, target_key_2, target_re_2)

# Trip angles taken directly from the keys
trip_angles_1 = get_trip_angles_from_key(target_key_1)
trip_angles_2 = get_trip_angles_from_key(target_key_2)

# Use common Cp scale across both plots
cp_abs_max = max(
    np.max(np.abs(run1["cp"][:-1])),
    np.max(np.abs(run2["cp"][:-1]))
)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

draw_cp_normal_bars(axes[0], run1, cp_abs_max, trip_angles=trip_angles_1)
draw_cp_normal_bars(axes[1], run2, cp_abs_max, trip_angles=trip_angles_2)

plt.tight_layout()
plt.show()

<font color = Red>

## C_P around cylinder with tripwire (Issue with tripwire not being on the circle)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Example
# =========================
target_key_1 = ("Trip", "0.94", "27")
target_re_1 = "Re250k"

target_key_2 = ("Trip", "0.94", "63")
target_re_2 = "Re250k"

def re_value(run_dict):
    return int(run_dict["re"].replace("Re", "").replace("k", ""))

def get_trip_angles_from_key(target_key):
    """
    Read trip-wire angles directly from target_key.
    Expected format for trip case: ("Trip", wire_diameter, angle)
    """
    if target_key[0] != "Trip":
        return []

    trip_angle = float(target_key[2])
    return [trip_angle, 360.0 - trip_angle]

def build_run_info_text(run_data, target_key):
    case_name = target_key[0]
    re_text = run_data["re"]

    if case_name == "Trip":
        trip_mm = target_key[1]
        angle_deg = target_key[2]
        return (
            f"Case: {case_name}\n"
            f"Re: {re_text}\n"
            f"Wire: {trip_mm} mm\n"
            f"Angle: {angle_deg}°"
        )
    else:
        return (
            f"Case: {case_name}\n"
            f"Re: {re_text}"
        )

def draw_trip_wire_arc(ax, center_angle_deg, radius=1.0,
                       arc_halfwidth_deg=2.5,
                       arc_radius_scale=1.04,
                       color="darkorange",
                       lw=4,
                       show_center_dot=True):
    arc_angles_deg = np.linspace(
        center_angle_deg - arc_halfwidth_deg,
        center_angle_deg + arc_halfwidth_deg,
        80
    )
    arc_angles_rad = np.deg2rad(arc_angles_deg)

    r_arc = radius * arc_radius_scale

    x_arc = -r_arc * np.cos(arc_angles_rad)
    y_arc =  r_arc * np.sin(arc_angles_rad)

    ax.plot(
        x_arc, y_arc,
        color=color,
        lw=lw,
        solid_capstyle="round",
        zorder=6
    )

    if show_center_dot:
        ang_rad = np.deg2rad(center_angle_deg)
        xt = -r_arc * np.cos(ang_rad)
        yt =  r_arc * np.sin(ang_rad)

        ax.plot(
            xt, yt,
            marker="o",
            ms=6,
            mec="black",
            mfc="gold",
            zorder=7
        )

def draw_cp_normal_bars(ax, run_data, cp_abs_max, trip_angles=None, info_text=None,
                        radius=1.0, max_bar_length=0.35,
                        show_trip_arc=True, trip_arc_halfwidth_deg=2.5):
    theta_deg = np.array(run_data["theta"][:-1])   # drop repeated 360° point
    cp_vals = np.array(run_data["cp"][:-1])
    theta_rad = np.deg2rad(theta_deg)

    scale = max_bar_length / cp_abs_max if cp_abs_max != 0 else 1.0

    # Cylinder surface coordinates
    x = -radius * np.cos(theta_rad)
    y =  radius * np.sin(theta_rad)

    # Outward unit normals
    nx = -np.cos(theta_rad)
    ny =  np.sin(theta_rad)

    # Cylinder outline
    phi = np.linspace(0, 2*np.pi, 400)
    ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.5)

    # Draw Cp bars
    for xs, ys, nxi, nyi, cp in zip(x, y, nx, ny, cp_vals):
        bar_len = scale * cp
        xe = xs + bar_len * nxi
        ye = ys + bar_len * nyi

        color = "crimson" if cp >= 0 else "royalblue"
        ax.plot([xs, xe], [ys, ye], color=color, lw=2.5, solid_capstyle="round")
        ax.plot(xs, ys, "ko", ms=3)

    # Plot trip-wire arcs
    if show_trip_arc and trip_angles:
        for ang in trip_angles:
            draw_trip_wire_arc(
                ax,
                center_angle_deg=ang,
                radius=radius,
                arc_halfwidth_deg=trip_arc_halfwidth_deg,
                arc_radius_scale=1.04,
                color="darkorange",
                lw=4,
                show_center_dot=True
            )

    # Angle labels every 30 degrees
    for xs, ys, ang in zip(x, y, theta_deg):
        if ang % 30 == 0:
            ax.text(
                1.12 * xs,
                1.12 * ys,
                f"{int(ang)}°",
                fontsize=8,
                ha="center",
                va="center"
            )

    # Info box
    if info_text is not None:
        ax.text(
            0.03, 0.97,
            info_text,
            transform=ax.transAxes,
            fontsize=10,
            ha="left",
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

    ax.set_aspect("equal")
    ax.set_xlim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_ylim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True)
    ax.set_title(f'{run_data["file_name"]}')

def get_filtered_run(grouped_results, target_key, target_re):
    if target_key not in grouped_results:
        raise ValueError(f"No files found for group {target_key}")

    runs = sorted(grouped_results[target_key], key=re_value)

    if target_re is not None:
        runs = [run for run in runs if run["re"] == target_re]

    if not runs:
        raise ValueError(f"No files found for group {target_key} with Reynolds number {target_re}")

    if len(runs) > 1:
        raise ValueError(
            f"More than one run matched {target_key} and {target_re}. "
            f"Expected exactly one."
        )

    return runs[0]

# Get the two selected runs
run1 = get_filtered_run(grouped_results, target_key_1, target_re_1)
run2 = get_filtered_run(grouped_results, target_key_2, target_re_2)

# Trip angles taken directly from the keys
trip_angles_1 = get_trip_angles_from_key(target_key_1)
trip_angles_2 = get_trip_angles_from_key(target_key_2)

# Info text
info_text_1 = build_run_info_text(run1, target_key_1)
info_text_2 = build_run_info_text(run2, target_key_2)

# Use common Cp scale across both plots
cp_abs_max = max(
    np.max(np.abs(run1["cp"][:-1])),
    np.max(np.abs(run2["cp"][:-1]))
)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

draw_cp_normal_bars(
    axes[0],
    run1,
    cp_abs_max,
    trip_angles=trip_angles_1,
    info_text=info_text_1
)

draw_cp_normal_bars(
    axes[1],
    run2,
    cp_abs_max,
    trip_angles=trip_angles_2,
    info_text=info_text_2
)

plt.tight_layout()
plt.show()

## All runs, combined, without Clean

### Each graph represents a run, that being where tripwire thickness and angle stay the same, and the reynolds number changes. These are all plotted in one graph, to illustrate the differences between each reynolds number

In [ ]:
for group_key, runs in grouped_results.items():
    if group_key[0] != "Trip":
        continue

    case, trip_mm, angle = group_key

    plt.figure(figsize=(9, 6))

    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(runs, key=re_value):
        plt.plot(
            run_data["theta"],
            run_data["cp"],
            "o-",
            label=run_data["re"]
        )

    plt.xlabel("Angle [deg]")
    plt.ylabel("$C_p$")
    plt.title(f"Trip wire {trip_mm} mm at A{angle}")
    plt.grid(True)
    plt.ylim(-4, 1.5)
    plt.legend(title="Re")


# C_Ds vs Reynolds Number 

### Grouping code for plotting C_D vs Reynolds number

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "file": file_path.name,
        "re": int(parts[1].replace("Re", "").replace("k", "")) * 1000,
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = float(parts[3])
        meta["angle"] = int(parts[4].replace("A", ""))

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

results = []

for FILE_PATH in sorted(DATA_FOLDER.glob("*.lvm")):
    meta = parse_filename(FILE_PATH)
    df = read_lvm_file(FILE_PATH)

    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    p1_mean = layer1.mean(axis=0)
    p2_mean = layer2.mean(axis=0)

    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    q_inf_mean = df.iloc[:, 63].mean()   # pitot channel

    cp1_mean = p1_mean / q_inf_mean
    cp2_mean = p2_mean / q_inf_mean

    cp_collapsed = np.empty(theta_collapsed.size)
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    cp_collapsed_closed = np.r_[cp_collapsed, cp_collapsed[0]]

    theta_rad_closed = np.deg2rad(theta_collapsed_closed)

    # positive sign to match your current angle convention
    C_D = 0.5 * np.trapz(cp_collapsed_closed * np.cos(theta_rad_closed), theta_rad_closed)
    C_L = 0.5 * np.trapz(cp_collapsed_closed * np.sin(theta_rad_closed), theta_rad_closed)

    results.append({
        "file": meta["file"],
        "Re": meta["re"],
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "C_D": C_D,
        "C_L": C_L
    })

results_df = pd.DataFrame(results)

# -----------------------------
# Build configuration labels
# -----------------------------
def config_label(row):
    if row["case"] == "Clean":
        return "Clean"
    return f'Trip {row["trip_mm"]:.2f} mm A{int(row["angle"])}'

results_df["config"] = results_df.apply(config_label, axis=1)

### Table with values

In [ ]:
print(results_df.sort_values(["case", "trip_mm", "angle", "Re"]).to_string(index=False))

### Specific run C_D

In [ ]:
selected_case = "Trip"
selected_trip = 0.94
selected_angle = 69

run_df = results_df[
    (results_df["case"] == selected_case) &
    (results_df["trip_mm"] == selected_trip) &
    (results_df["angle"] == selected_angle)
].sort_values("Re")

if run_df.empty:
    print("No matching run found.")
else:
    plt.figure(figsize=(9, 6))
    plt.plot(run_df["Re"], run_df["C_D"], "o-", label=f"A{int(selected_angle)}")

    plt.xlabel("Reynolds number")
    plt.ylabel("$C_D$")
    plt.title(f"$C_D$ vs Reynolds number - {selected_case}, {selected_trip:.2f} mm")
    plt.grid(True)
    plt.ylim(0.5, 1.5)
    plt.legend()
    plt.show()



### All C_D vs Reynolds - one plot

In [ ]:
plt.figure(figsize=(10, 6))

for config, group in results_df.groupby("config"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_D"], "o-", label=config)

plt.xlabel("Reynolds number")
plt.ylabel("$C_D$")
plt.title("$C_D$ vs Reynolds number")
plt.grid(True)
plt.legend()
plt.show()

### Clean cylinder C_D vs Reynolds number

In [ ]:

# -----------------------------
# Plot 1: Clean only
# -----------------------------
clean_df = results_df[results_df["case"] == "Clean"].sort_values("Re")

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    clean_df["Re"],
    clean_df["C_D"],
    "o-",
    label="Smooth",
    color="royalblue",
    linewidth=1.8,
    markersize=6
)

# Add C_D value labels
for _, row in clean_df.iterrows():
    ax.annotate(
        f"{row['C_D']:.2f}",                 # label text
        xy=(row["Re"], row["C_D"]),          # point being labelled
        xytext=(0, 10),                      # offset above marker
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        color="black",
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="0.75",
            alpha=0.85
        )
    )

ax.set_xlabel("Reynolds number")
ax.set_ylabel("$C_D$")
ax.set_title("$C_D$ vs Reynolds number - Smooth cylinder day 1")
ax.set_ylim(0.35, 1.5)
ax.grid(True)
ax.legend()

plt.show()

### 0.15mm trip wire, C_D vs Reynolds number

In [ ]:
# -----------------------------
# Plot 2: 0.15 mm trip only
# one line per angle
# -----------------------------
trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.15)
]

plt.figure(figsize=(9, 6))
for angle, group in trip015_df.groupby("angle"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_D"], "o-", label=f"A{int(angle)}")

plt.xlabel("Reynolds number")
plt.ylabel("$C_D$")
plt.title("$C_D$ vs Reynolds number - Trip wire 0.15 mm")
plt.grid(True)
plt.legend(title="Angle")
plt.show()

### 0.94mm tripwire, C_D vs Reynolds number

In [ ]:

# -----------------------------
# Plot 3: 0.94 mm trip only
# one line per angle
# -----------------------------
trip094_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.94)
]

plt.figure(figsize=(9, 6))
for angle, group in trip094_df.groupby("angle"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_D"], "o-", label=f"A{int(angle)}")

plt.xlabel("Reynolds number")
plt.ylabel("$C_D$")
plt.title("$C_D$ vs Reynolds number - Trip wire 0.94 mm")
plt.grid(True)
plt.legend(title="Angle")
plt.show()

### 0.94mm tripwire AND Clean run, C_D vs Reynolds number

In [ ]:
# -----------------------------
# Plot: 0.94 mm trip + clean cylinder
# -----------------------------

# Clean cylinder
clean_df = results_df[
    results_df["case"] == "Clean"
].sort_values("Re")

# 0.94 mm trip wire
trip094_df = results_df[
    (results_df["case"] == "Trip") & 
    (results_df["trip_mm"] == 0.94)
]

plt.figure(figsize=(9, 6))

# Plot clean cylinder first
plt.plot(
    clean_df["Re"],
    clean_df["C_D"],
    "o-",
    color="black",
    linewidth=2.5,
    markersize=6,
    label="Clean"
)

# Plot one line per trip-wire angle
for angle, group in trip094_df.groupby("angle"):
    group = group.sort_values("Re")
    plt.plot(
        group["Re"],
        group["C_D"],
        "o-",
        label=f"A{int(angle)}"
    )

plt.xlabel("Reynolds number")
plt.ylabel("$C_D$")
plt.ylim(0.35, 1.5)
plt.title("$C_D$ vs Reynolds number - Clean cylinder and 0.94 mm trip wire")
plt.grid(True)
plt.legend(title="Configuration")
plt.show()

### Two runs in same plot

In [ ]:

trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"].isin([0.15, 0.94])) & (results_df["angle"] == 69) 
]

plt.figure(figsize=(9, 6))
for trip_mm, group in trip015_df.groupby("trip_mm"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_D"], "o-", label=f"Trip {trip_mm:.2f} mm", color="crimson" if trip_mm == 0.94 else "royalblue")

plt.xlabel("Reynolds number")
plt.ylabel("$C_D$")
plt.title("$C_D$ vs Reynolds number - Trip wire 0.15 mm")
plt.grid(True)
plt.legend(title="Angle")
plt.show()

<font color = Red>

### 3D plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "file": file_path.name,
        "re": int(parts[1].replace("Re", "").replace("k", "")) * 1000,
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = float(parts[3])
        meta["angle"] = int(parts[4].replace("A", ""))

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

results = []

for FILE_PATH in sorted(DATA_FOLDER.glob("*.lvm")):
    meta = parse_filename(FILE_PATH)
    df = read_lvm_file(FILE_PATH)

    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    p1_mean = layer1.mean(axis=0)
    p2_mean = layer2.mean(axis=0)

    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    q_inf_mean = df.iloc[:, 63].mean()   # pitot channel

    cp1_mean = p1_mean / q_inf_mean
    cp2_mean = p2_mean / q_inf_mean

    cp_collapsed = np.empty(theta_collapsed.size)
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    cp_collapsed_closed = np.r_[cp_collapsed, cp_collapsed[0]]

    theta_rad_closed = np.deg2rad(theta_collapsed_closed)

    # sign chosen to match your current angle convention
    C_D = 0.5 * np.trapz(cp_collapsed_closed * np.cos(theta_rad_closed), theta_rad_closed)
    C_L = 0.5 * np.trapz(cp_collapsed_closed * np.sin(theta_rad_closed), theta_rad_closed)

    results.append({
        "file": meta["file"],
        "Re": meta["re"],
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "C_D": C_D,
        "C_L": C_L
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values(["case", "trip_mm", "angle", "Re"]).to_string(index=False))

# -----------------------------
# Plot 1: Clean only (2D)
# -----------------------------
clean_df = results_df[results_df["case"] == "Clean"].sort_values("Re")

plt.figure(figsize=(9, 6))
plt.plot(clean_df["Re"], clean_df["C_D"], "o-", label="Clean")
plt.xlabel("Reynolds number")
plt.ylabel("$C_D$")
plt.title("$C_D$ vs Reynolds number - Clean cylinder")
plt.grid(True)
plt.legend()
plt.show()

# -----------------------------
# Plot 2: 0.15 mm trip only (3D)
# x = Re, y = angle, z = C_D
# -----------------------------
trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.15)
]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for angle, group in trip015_df.groupby("angle"):
    group = group.sort_values("Re")
    x = group["Re"].to_numpy()
    y = np.full_like(x, angle)
    z = group["C_D"].to_numpy()

    ax.plot(x, y, z, marker="o", label=f"A{int(angle)}")

ax.set_xlabel("Reynolds number")
ax.set_ylabel("Trip-wire angle [deg]")
ax.set_zlabel("$C_D$")
ax.set_title("$C_D$ vs Reynolds number and angle - Trip wire 0.15 mm")
ax.legend(title="Angle")

plt.show()

# -----------------------------
# Plot 3: 0.94 mm trip only (3D)
# x = Re, y = angle, z = C_D
# -----------------------------
trip094_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.94)
]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for angle, group in trip094_df.groupby("angle"):
    group = group.sort_values("Re")
    x = group["Re"].to_numpy()
    y = np.full_like(x, angle)
    z = group["C_D"].to_numpy()

    ax.plot(x, y, z, marker="o", label=f"A{int(angle)}")

ax.set_xlabel("Reynolds number")
ax.set_ylabel("Trip-wire angle [deg]")
ax.set_zlabel("$C_D$")
ax.set_title("$C_D$ vs Reynolds number and angle - Trip wire 0.94 mm")
ax.legend(title="Angle")

plt.show()

<font color = Red>

### Contour plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "file": file_path.name,
        "re": int(parts[1].replace("Re", "").replace("k", "")) * 1000,
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = float(parts[3])
        meta["angle"] = int(parts[4].replace("A", ""))

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

results = []

for FILE_PATH in sorted(DATA_FOLDER.glob("*.lvm")):
    meta = parse_filename(FILE_PATH)
    df = read_lvm_file(FILE_PATH)

    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    p1_mean = layer1.mean(axis=0)
    p2_mean = layer2.mean(axis=0)

    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    q_inf_mean = df.iloc[:, 63].mean()

    cp1_mean = p1_mean / q_inf_mean
    cp2_mean = p2_mean / q_inf_mean

    cp_collapsed = np.empty(theta_collapsed.size)
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    cp_collapsed_closed = np.r_[cp_collapsed, cp_collapsed[0]]

    theta_rad_closed = np.deg2rad(theta_collapsed_closed)

    C_D = 0.5 * np.trapz(cp_collapsed_closed * np.cos(theta_rad_closed), theta_rad_closed)
    C_L = 0.5 * np.trapz(cp_collapsed_closed * np.sin(theta_rad_closed), theta_rad_closed)

    results.append({
        "file": meta["file"],
        "Re": meta["re"],
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "C_D": C_D,
        "C_L": C_L
    })

results_df = pd.DataFrame(results)

# -----------------------------
# Contour plot function
# -----------------------------
def plot_cd_contour(df_trip, trip_mm):
    # Build a grid: rows = angle, columns = Re, values = C_D
    pivot = df_trip.pivot(index="angle", columns="Re", values="C_D")
    pivot = pivot.sort_index().sort_index(axis=1)

    X, Y = np.meshgrid(pivot.columns.to_numpy(), pivot.index.to_numpy())
    Z = pivot.to_numpy()

    plt.figure(figsize=(9, 6))

    # filled contour
    cf = plt.contourf(X, Y, Z, levels=15)
    plt.colorbar(cf, label="$C_D$")

    # contour lines on top
    cs = plt.contour(X, Y, Z, levels=15)
    plt.clabel(cs, inline=True, fontsize=8)

    # mark actual data points
    plt.plot(X.flatten(), Y.flatten(), "ko", markersize=4)

    plt.xlabel("Reynolds number")
    plt.ylabel("Trip-wire angle [deg]")
    plt.title(f"$C_D$ contour plot - Trip wire {trip_mm:.2f} mm")
    plt.grid(True, alpha=0.3)
    plt.show()

# -----------------------------
# 0.15 mm contour
# -----------------------------
trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.15)
]

plot_cd_contour(trip015_df, 0.15)

# -----------------------------
# 0.94 mm contour
# -----------------------------
trip094_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.94)
]

plot_cd_contour(trip094_df, 0.94)

# C_L vs Reynolds number

### Table with values

In [ ]:
print(results_df.sort_values(["case", "trip_mm", "angle", "Re"]).to_string(index=False))

In [ ]:
selected_case = "Trip"
selected_trip = 0.15
selected_angle = 57

run_df = results_df[
    (results_df["case"] == selected_case) &
    (results_df["trip_mm"] == selected_trip) &
    (results_df["angle"] == selected_angle)
].sort_values("Re")

if run_df.empty:
    print("No matching run found.")
else:
    plt.figure(figsize=(9, 6))
    plt.plot(run_df["Re"], run_df["C_L"], "o-", label=f"A{int(selected_angle)}")

    plt.xlabel("Reynolds number")
    plt.ylabel("$C_L$")
    plt.title(f"$C_L$ vs Reynolds number - {selected_case}, {selected_trip:.2f} mm")
    plt.grid(True)
    plt.legend()
    plt.show()

### All C_L vs Reynolds

In [ ]:
plt.figure(figsize=(10, 6))

for config, group in results_df.groupby("config"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_L"], "o-", label=config)
    
plt.xlabel("Reynolds number")
plt.ylim(-0.5, 0.5)
plt.ylabel("$C_L$")
plt.title("$C_L$ vs Reynolds number")
plt.grid(True)
plt.legend()
plt.show()

### Clean Cylinder, C_L vs Reynolds

In [ ]:
clean_df = results_df[results_df["case"] == "Clean"].sort_values("Re")

plt.figure(figsize=(9, 6))
plt.plot(clean_df["Re"], clean_df["C_L"], "o-", label="Smooth", color="cyan")
plt.xlabel("Reynolds number")
plt.ylim(-0.05, 0.05)
plt.ylabel("$C_L$")
plt.title("$C_L$ vs Reynolds number - Smooth cylinder day 1")
plt.grid(True)
plt.legend()
plt.show()

### 0.15mm trip wire, C_L vs Reynolds number

In [ ]:
trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.15)
]

plt.figure(figsize=(9, 6))
for angle, group in trip015_df.groupby("angle"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_L"], "o-", label=f"A{int(angle)}")

plt.xlabel("Reynolds number")
plt.ylabel("$C_L$")
plt.title("$C_L$ vs Reynolds number - Trip wire 0.15 mm")
plt.grid(True)
plt.legend(title="Angle")
plt.show()

### 0.94mm tripwire, C_L vs Reynolds number

In [ ]:
trip094_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.94)
]

plt.figure(figsize=(9, 6))
for angle, group in trip094_df.groupby("angle"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_L"], "o-", label=f"A{int(angle)}")

plt.xlabel("Reynolds number")
plt.ylabel("$C_L$")
plt.title("$C_L$ vs Reynolds number - Trip wire 0.94 mm")
plt.grid(True)
plt.legend(title="Angle")
plt.show()

### Two plots same figure

In [ ]:

trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"].isin([0.15, 0.94])) & (results_df["angle"] == 69) 
]

plt.figure(figsize=(9, 6))
for trip_mm, group in trip015_df.groupby("trip_mm"):
    group = group.sort_values("Re")
    plt.plot(group["Re"], group["C_L"], "o-", label=f"Trip {trip_mm:.2f} mm", color="crimson" if trip_mm == 0.94 else "royalblue")

plt.xlabel("Reynolds number")
plt.ylabel("$C_L$")
plt.title("$C_L$ vs Reynolds number - Trip wire 0.15 mm")
plt.grid(True)
plt.legend(title="Angle")
plt.show()

<font color = Red>

### 3D plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "file": file_path.name,
        "re": int(parts[1].replace("Re", "").replace("k", "")) * 1000,
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = float(parts[3])
        meta["angle"] = int(parts[4].replace("A", ""))

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

results = []

for FILE_PATH in sorted(DATA_FOLDER.glob("*.lvm")):
    meta = parse_filename(FILE_PATH)
    df = read_lvm_file(FILE_PATH)

    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    p1_mean = layer1.mean(axis=0)
    p2_mean = layer2.mean(axis=0)

    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    q_inf_mean = df.iloc[:, 63].mean()   # pitot channel

    cp1_mean = p1_mean / q_inf_mean
    cp2_mean = p2_mean / q_inf_mean

    cp_collapsed = np.empty(theta_collapsed.size)
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    cp_collapsed_closed = np.r_[cp_collapsed, cp_collapsed[0]]

    theta_rad_closed = np.deg2rad(theta_collapsed_closed)

    # sign chosen to match your current angle convention
    C_D = 0.5 * np.trapz(cp_collapsed_closed * np.cos(theta_rad_closed), theta_rad_closed)
    C_L = 0.5 * np.trapz(cp_collapsed_closed * np.sin(theta_rad_closed), theta_rad_closed)

    results.append({
        "file": meta["file"],
        "Re": meta["re"],
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "C_D": C_D,
        "C_L": C_L
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values(["case", "trip_mm", "angle", "Re"]).to_string(index=False))

# -----------------------------
# Plot 1: Clean only (2D)
# -----------------------------
clean_df = results_df[results_df["case"] == "Clean"].sort_values("Re")

plt.figure(figsize=(9, 6))
plt.plot(clean_df["Re"], clean_df["C_L"], "o-", label="Clean")
plt.xlabel("Reynolds number")
plt.ylabel("$C_L$")
plt.title("$C_L$ vs Reynolds number - Clean cylinder")
plt.grid(True)
plt.legend()
plt.show()

# -----------------------------
# Plot 2: 0.15 mm trip only (3D)
# x = Re, y = angle, z = C_L
# -----------------------------
trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.15)
]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for angle, group in trip015_df.groupby("angle"):
    group = group.sort_values("Re")
    x = group["Re"].to_numpy()
    y = np.full_like(x, angle)
    z = group["C_L"].to_numpy()

    ax.plot(x, y, z, marker="o", label=f"A{int(angle)}")

ax.set_xlabel("Reynolds number")
ax.set_ylabel("Trip-wire angle [deg]")
ax.set_zlabel("$C_L$")
ax.set_title("$C_L$ vs Reynolds number and angle - Trip wire 0.15 mm")
ax.legend(title="Angle")

plt.show()

# -----------------------------
# Plot 3: 0.94 mm trip only (3D)
# x = Re, y = angle, z = C_L
# -----------------------------
trip094_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.94)
]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for angle, group in trip094_df.groupby("angle"):
    group = group.sort_values("Re")
    x = group["Re"].to_numpy()
    y = np.full_like(x, angle)
    z = group["C_L"].to_numpy()

    ax.plot(x, y, z, marker="o", label=f"A{int(angle)}")

ax.set_xlabel("Reynolds number")
ax.set_ylabel("Trip-wire angle [deg]")
ax.set_zlabel("$C_L$")
ax.set_title("$C_L$ vs Reynolds number and angle - Trip wire 0.94 mm")
ax.legend(title="Angle")

plt.show()

<font color = Red>

### Contour plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "file": file_path.name,
        "re": int(parts[1].replace("Re", "").replace("k", "")) * 1000,
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = float(parts[3])
        meta["angle"] = int(parts[4].replace("A", ""))

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

results = []

for FILE_PATH in sorted(DATA_FOLDER.glob("*.lvm")):
    meta = parse_filename(FILE_PATH)
    df = read_lvm_file(FILE_PATH)

    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    p1_mean = layer1.mean(axis=0)
    p2_mean = layer2.mean(axis=0)

    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    q_inf_mean = df.iloc[:, 63].mean()

    cp1_mean = p1_mean / q_inf_mean
    cp2_mean = p2_mean / q_inf_mean

    cp_collapsed = np.empty(theta_collapsed.size)
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    cp_collapsed_closed = np.r_[cp_collapsed, cp_collapsed[0]]

    theta_rad_closed = np.deg2rad(theta_collapsed_closed)

    C_D = 0.5 * np.trapz(cp_collapsed_closed * np.cos(theta_rad_closed), theta_rad_closed)
    C_L = 0.5 * np.trapz(cp_collapsed_closed * np.sin(theta_rad_closed), theta_rad_closed)

    results.append({
        "file": meta["file"],
        "Re": meta["re"],
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "C_D": C_D,
        "C_L": C_L
    })

results_df = pd.DataFrame(results)

# -----------------------------
# Contour plot function
# -----------------------------
def plot_cl_contour(df_trip, trip_mm):
    # Build a grid: rows = angle, columns = Re, values = C_L
    pivot = df_trip.pivot(index="angle", columns="Re", values="C_L")
    pivot = pivot.sort_index().sort_index(axis=1)

    X, Y = np.meshgrid(pivot.columns.to_numpy(), pivot.index.to_numpy())
    Z = pivot.to_numpy()

    plt.figure(figsize=(9, 6))

    # filled contour
    cf = plt.contourf(X, Y, Z, levels=15)
    plt.colorbar(cf, label="$C_L$")

    # contour lines on top
    cs = plt.contour(X, Y, Z, levels=15)
    plt.clabel(cs, inline=True, fontsize=8)

    # mark actual data points
    plt.plot(X.flatten(), Y.flatten(), "ko", markersize=4)

    plt.xlabel("Reynolds number")
    plt.ylabel("Trip-wire angle [deg]")
    plt.title(f"$C_L$ contour plot - Trip wire {trip_mm:.2f} mm")
    plt.grid(True, alpha=0.3)
    plt.show()

# -----------------------------
# 0.15 mm contour
# -----------------------------
trip015_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.15)
]

plot_cl_contour(trip015_df, 0.15)

# -----------------------------
# 0.94 mm contour
# -----------------------------
trip094_df = results_df[
    (results_df["case"] == "Trip") & (results_df["trip_mm"] == 0.94)
]

plot_cl_contour(trip094_df, 0.94)

# Strouhal number

#### First, we isolate the time from a specific file

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

FILE_PATH = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM\Cyl100_Re100k_Clean.lvm")

EXTRA_NAMES = ["pitot", "AoA", "F", "alpha", "P_m", "V_tunnel", "rho", "time"]

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")
    return {
        "re": parts[1],
        "case": parts[2]
    }

# Read just this one file
df = read_lvm_file(FILE_PATH)
meta = parse_filename(FILE_PATH)

# Extract the extra channels
extra_data = df.iloc[:, 63:71].copy()
extra_data.columns = EXTRA_NAMES

# Plot each variable for this one run
# for col in EXTRA_NAMES:
#     plt.figure(figsize=(10, 5))
#     plt.plot(extra_data.index, extra_data[col])

#     plt.xlabel("Sample number")
#     plt.ylabel(col)
#     plt.title(f"{col} for {meta['re']} {meta['case']}")
#     plt.grid(True)

# plt.show()

# Make one summary table for this file
summary_rows = []

for col in EXTRA_NAMES:
    series = extra_data[col]
    summary_rows.append({
        "variable": col,
        "mean": series.mean(),
        "std": series.std(),
        "min": series.min(),
        "max": series.max()
    })

summary_df = pd.DataFrame(summary_rows)

# print("\nSummary table for selected file:")
# print(summary_df)



for col in EXTRA_NAMES:
    if col != "time":
        continue
    print(extra_data[col])


### C_L not as a mean value

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILE_PATH = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM\Cyl100_Re100k_Clean.lvm")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

EXTRA_NAMES = ["pitot", "AoA", "F", "alpha", "P_m", "V_tunnel", "rho", "time"]

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

df = read_lvm_file(FILE_PATH)

# Pressure tap data: shape = (n_samples, 30)
layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

# Extra channels
extra_data = df.iloc[:, 63:71].copy()
extra_data.columns = EXTRA_NAMES

# Time and pitot as time series
time = extra_data["time"].to_numpy()
q_inf = extra_data["pitot"].to_numpy()

# Avoid division by zero
eps = 1e-12
q_inf = np.where(np.abs(q_inf) < eps, np.nan, q_inf)

# Cp(t, theta) for every sample
cp1 = layer1 / q_inf[:, None]
cp2 = layer2 / q_inf[:, None]

# Angular positions
theta_collapsed = np.arange(0.0, 360.0, 6.0)   # 60 positions
theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
theta_rad_closed = np.deg2rad(theta_collapsed_closed)

n_samples = df.shape[0]
cp_collapsed = np.empty((n_samples, theta_collapsed.size))

# Interleave the two layers exactly as before
cp_collapsed[:, 0::2] = cp2
cp_collapsed[:, 1::2] = cp1

# Close the curve in theta for integration
cp_collapsed_closed = np.hstack([cp_collapsed, cp_collapsed[:, [0]]])

# Compute C_L(t) and C_D(t)
C_L_t = 0.5 * np.trapz(cp_collapsed_closed * np.sin(theta_rad_closed), theta_rad_closed, axis=1)
C_D_t = 0.5 * np.trapz(cp_collapsed_closed * np.cos(theta_rad_closed), theta_rad_closed, axis=1)

# Put in dataframe
results_t = pd.DataFrame({
    "time": time,
    "C_L": C_L_t,
    "C_D": C_D_t
})

print(results_t.head())

# Plot C_L(t)
plt.figure(figsize=(10, 5))
plt.plot(results_t["time"], results_t["C_L"])
plt.xlabel("Time [s]")
plt.ylabel("C_L")
plt.title("C_L as a function of time")
plt.grid(True)
plt.show()

### Strouhal number calculation finder

#### Target Key (File path)

In [ ]:
FILE_PATH = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM\Cyl100_Re100k_Clean.lvm")

#### Code

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

EXTRA_NAMES = ["pitot", "AoA", "F", "alpha", "P_m", "V_tunnel", "rho", "time"]

D = 0.100   # cylinder diameter [m] - change if needed

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

df = read_lvm_file(FILE_PATH)

# Pressure tap data
layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)   # shape (n_samples, 30)
layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)   # shape (n_samples, 30)

# Extra channels
extra_data = df.iloc[:, 63:71].copy()
extra_data.columns = EXTRA_NAMES

time = extra_data["time"].to_numpy()
q_inf = extra_data["pitot"].to_numpy()
U_tunnel = extra_data["V_tunnel"].to_numpy()

# Avoid division by zero
eps = 1e-12
q_inf = np.where(np.abs(q_inf) < eps, np.nan, q_inf)

# Cp for every sample
cp1 = layer1 / q_inf[:, None]
cp2 = layer2 / q_inf[:, None]

# Angular positions
theta_collapsed = np.arange(0.0, 360.0, 6.0)   # 60 positions
theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
theta_rad_closed = np.deg2rad(theta_collapsed_closed)

# Interleave tap rows into one circumferential Cp distribution per sample
n_samples = df.shape[0]
cp_collapsed = np.empty((n_samples, theta_collapsed.size))
cp_collapsed[:, 0::2] = cp2
cp_collapsed[:, 1::2] = cp1

# Close the loop for integration
cp_collapsed_closed = np.hstack([cp_collapsed, cp_collapsed[:, [0]]])

# Compute C_L(t) and C_D(t)
C_L_t = 0.5 * np.trapz(
    cp_collapsed_closed * np.sin(theta_rad_closed),
    theta_rad_closed,
    axis=1
)
C_D_t = 0.5 * np.trapz(
    cp_collapsed_closed * np.cos(theta_rad_closed),
    theta_rad_closed,
    axis=1
)

# Build results table
results_t = pd.DataFrame({
    "time": time,
    "C_L": C_L_t,
    "C_D": C_D_t,
    "V_tunnel": U_tunnel
})

# Remove bad rows
results_t = results_t.replace([np.inf, -np.inf], np.nan).dropna(subset=["time", "C_L", "V_tunnel"])

# Optional: remove initial transient (first 10%)
start_idx = int(0.10 * len(results_t))
results_fft = results_t.iloc[start_idx:].copy()

# Extract arrays for FFT
time_fft = results_fft["time"].to_numpy()
cl_fft = results_fft["C_L"].to_numpy()

# Remove mean from C_L(t)
cd_fluct = cl_fft - np.mean(cl_fft)

# Sampling interval from time series
dt = np.mean(np.diff(time_fft))

# FFT with Hann window
n = len(cl_fluct)
window = np.hanning(n)
cl_windowed = cl_fluct * window

freqs = np.fft.rfftfreq(n, d=dt)
fft_vals = np.fft.rfft(cl_windowed)
amplitude = np.abs(fft_vals)

# Remove zero-frequency component
freqs_nozero = freqs[1:]
amplitude_nozero = amplitude[1:]

# Optional: restrict search range to avoid drift/high-frequency noise
# Adjust bounds if needed
f_min = 1.0
f_max = 500.0

band_mask = (freqs_nozero >= f_min) & (freqs_nozero <= f_max)

freqs_band = freqs_nozero[band_mask]
amplitude_band = amplitude_nozero[band_mask]

dominant_index = np.argmax(amplitude_band)
f_shedding = freqs_band[dominant_index]

# Mean tunnel velocity
U_mean = results_fft["V_tunnel"].mean()

# Strouhal number
St = f_shedding * D / U_mean

#### Plot

In [ ]:
print(f"File: {FILE_PATH.name}")
print(f"Number of samples used for FFT: {n}")
print(f"Mean velocity U = {U_mean:.4f} m/s")
print(f"Dominant shedding frequency f = {f_shedding:.4f} Hz")
print(f"Strouhal number St = {St:.4f}")
print(f"Drag coefficient, C_D mean = {results_t['C_D'].mean():.4f}, std = {results_t['C_D'].std():.4f}")

# Plot C_L(t)
plt.figure(figsize=(10, 5))
plt.plot(results_t["time"], results_t["C_L"], color = "orange")
plt.xlabel("Time [s]")
plt.ylabel("C_L")
plt.title("C_L(t)")
plt.grid(True)

# Plot FFT spectrum
plt.figure(figsize=(10, 5))
plt.plot(freqs_nozero, amplitude_nozero, label="FFT amplitude", color = "orange")
plt.axvline(f_shedding, linestyle="dotted", label=f"Peak = {f_shedding:.3f} Hz", color = "indigo")
plt.xlim(0, min(200, freqs_nozero.max()))  # adjust if needed
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("FFT of C_L(t)")
plt.grid(True)
plt.legend()

plt.show()

<font color = Red>

## FFT of drag (Issues with the 10% removal) <-- consider scrapping

### Target Key (File path)

In [ ]:
FILE_PATH = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM\Cyl100_Re100k_Clean.lvm")

### Code

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

EXTRA_NAMES = ["pitot", "AoA", "F", "alpha", "P_m", "V_tunnel", "rho", "time"]

D = 0.100   # cylinder diameter [m] - change if needed

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

df = read_lvm_file(FILE_PATH)

# Pressure tap data
layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)   # shape (n_samples, 30)
layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)   # shape (n_samples, 30)

# Extra channels
extra_data = df.iloc[:, 63:71].copy()
extra_data.columns = EXTRA_NAMES

time = extra_data["time"].to_numpy()
q_inf = extra_data["pitot"].to_numpy()
U_tunnel = extra_data["V_tunnel"].to_numpy()

# Avoid division by zero
eps = 1e-12
q_inf = np.where(np.abs(q_inf) < eps, np.nan, q_inf)

# Cp for every sample
cp1 = layer1 / q_inf[:, None]
cp2 = layer2 / q_inf[:, None]

# Angular positions
theta_collapsed = np.arange(0.0, 360.0, 6.0)   # 60 positions
theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
theta_rad_closed = np.deg2rad(theta_collapsed_closed)

# Interleave tap rows into one circumferential Cp distribution per sample
n_samples = df.shape[0]
cp_collapsed = np.empty((n_samples, theta_collapsed.size))
cp_collapsed[:, 0::2] = cp2
cp_collapsed[:, 1::2] = cp1

# Close the loop for integration
cp_collapsed_closed = np.hstack([cp_collapsed, cp_collapsed[:, [0]]])

# Compute C_L(t) and C_D(t)
C_L_t = 0.5 * np.trapz(
    cp_collapsed_closed * np.sin(theta_rad_closed),
    theta_rad_closed,
    axis=1
)
C_D_t = 0.5 * np.trapz(
    cp_collapsed_closed * np.cos(theta_rad_closed),
    theta_rad_closed,
    axis=1
)

# Build results table
results_t = pd.DataFrame({
    "time": time,
    "C_L": C_L_t,
    "C_D": C_D_t,
    "V_tunnel": U_tunnel
})

# Optional: remove initial transient (first 10%)
start_idx = int(0.10 * len(results_t))
results_fft = results_t.iloc[start_idx:].copy()

# Extract arrays for FFT
time_fft = results_fft["time"].to_numpy()
cd_fft = results_fft["C_D"].to_numpy()

# Remove mean from C_D(t)
cd_fluct = cd_fft - np.mean(cd_fft)

# Sampling interval from time series
dt = np.mean(np.diff(time_fft))

# FFT with Hann window
n = len(cd_fluct)
window = np.hanning(n)
cd_windowed = cd_fluct * window

freqs = np.fft.rfftfreq(n, d=dt)
fft_vals = np.fft.rfft(cd_windowed)
amplitude = np.abs(fft_vals)

# Remove zero-frequency component
freqs_nozero = freqs[1:]
amplitude_nozero = amplitude[1:]

# Optional: restrict search range to avoid drift/high-frequency noise
# Adjust bounds if needed
f_min = 1.0
f_max = 500.0

band_mask = (freqs_nozero >= f_min) & (freqs_nozero <= f_max)

freqs_band = freqs_nozero[band_mask]
amplitude_band = amplitude_nozero[band_mask]

dominant_index = np.argmax(amplitude_band)
f_shedding = freqs_band[dominant_index]

# Mean tunnel velocity
U_mean = results_fft["V_tunnel"].mean()


### Plot

In [ ]:
print(f"File: {FILE_PATH.name}")
print(f"Number of samples used for FFT: {n}")
print(f"Mean velocity U = {U_mean:.4f} m/s")
print(f"Dominant shedding frequency f = {f_shedding:.4f} Hz")
print(f"Drag coefficient, C_D mean = {results_t['C_D'].mean():.4f}, std = {results_t['C_D'].std():.4f}")

# Plot C_D(t)
plt.figure(figsize=(10, 5))
plt.plot(results_t["time"], results_t["C_D"], color = "orange")
plt.xlabel("Time [s]")
plt.ylabel("C_D")
plt.title("C_D(t)")
plt.grid(True)

# Plot FFT spectrum
plt.figure(figsize=(10, 5))
plt.plot(freqs_nozero, amplitude_nozero, label="FFT amplitude", color = "orange")
plt.axvline(f_shedding, linestyle="dotted", label=f"Peak = {f_shedding:.3f} Hz", color = "indigo")
plt.xlim(0, min(200, freqs_nozero.max()))  # adjust if needed
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title("FFT of C_D(t)")
plt.grid(True)
plt.legend()

plt.show()


## 5 sections average method

del time op i 5, og lav fft af hver, og gennemsnit af de 5


### Target Key (File Path)

In [ ]:
FILE_PATH = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM\Cyl100_Re100k_Clean.lvm")


### Code

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

EXTRA_NAMES = ["pitot", "AoA", "F", "alpha", "P_m", "V_tunnel", "rho", "time"]

D = 0.100   # cylinder diameter [m]

# -------------------------
# Settings for segmented FFT
# -------------------------
N_SEGMENTS = 5          # try 5 first; compare with 10 later
REMOVE_FIRST_FRAC = 0.10
F_MIN = 1.0
F_MAX = 500.0

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

df = read_lvm_file(FILE_PATH)

# Pressure tap data
layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

# Extra channels
extra_data = df.iloc[:, 63:71].copy()
extra_data.columns = EXTRA_NAMES

time = extra_data["time"].to_numpy()
q_inf = extra_data["pitot"].to_numpy()
U_tunnel = extra_data["V_tunnel"].to_numpy()

# Avoid division by zero
eps = 1e-12
q_inf = np.where(np.abs(q_inf) < eps, np.nan, q_inf)

# Cp for every sample
cp1 = layer1 / q_inf[:, None]
cp2 = layer2 / q_inf[:, None]

# Angular positions
theta_collapsed = np.arange(0.0, 360.0, 6.0)   # 60 positions
theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
theta_rad_closed = np.deg2rad(theta_collapsed_closed)

# Interleave taps
n_samples = df.shape[0]
cp_collapsed = np.empty((n_samples, theta_collapsed.size))
cp_collapsed[:, 0::2] = cp2
cp_collapsed[:, 1::2] = cp1

# Close loop for integration
cp_collapsed_closed = np.hstack([cp_collapsed, cp_collapsed[:, [0]]])

# Compute C_L(t) and C_D(t)
C_L_t = 0.5 * np.trapz(
    cp_collapsed_closed * np.sin(theta_rad_closed),
    theta_rad_closed,
    axis=1
)
C_D_t = 0.5 * np.trapz(
    cp_collapsed_closed * np.cos(theta_rad_closed),
    theta_rad_closed,
    axis=1
)

# Build results table
results_t = pd.DataFrame({
    "time": time,
    "C_L": C_L_t,
    "C_D": C_D_t,
    "V_tunnel": U_tunnel
})

# Remove bad rows
results_t = results_t.replace([np.inf, -np.inf], np.nan).dropna(subset=["time", "C_L", "V_tunnel"])

# Remove initial transient
start_idx = int(REMOVE_FIRST_FRAC * len(results_t))
results_fft = results_t.iloc[start_idx:].copy()

time_fft = results_fft["time"].to_numpy()
cl_fft = results_fft["C_L"].to_numpy()
U_mean = results_fft["V_tunnel"].mean()

# Sampling interval
dt = np.mean(np.diff(time_fft))
fs = 1.0 / dt

# -------------------------
# Segmented FFT
# -------------------------
n_total = len(cl_fft)
segment_length = n_total // N_SEGMENTS

if segment_length < 10:
    raise ValueError("Too few samples per segment. Reduce N_SEGMENTS.")

# Truncate to exact multiple of segment_length
n_used = segment_length * N_SEGMENTS
cl_used = cl_fft[:n_used]
time_used = time_fft[:n_used]

# Reshape into segments
cl_segments = cl_used.reshape(N_SEGMENTS, segment_length)

# Frequency axis (same for each segment)
freqs = np.fft.rfftfreq(segment_length, d=dt)

# Store spectra
power_spectra = []

for seg in cl_segments:
    # Remove segment mean
    seg_fluct = seg - np.mean(seg)

    # Apply Hann window
    window = np.hanning(segment_length)
    seg_windowed = seg_fluct * window

    # FFT
    fft_vals = np.fft.rfft(seg_windowed)

    # Power spectrum
    power = np.abs(fft_vals)**2
    power_spectra.append(power)

power_spectra = np.array(power_spectra)

# Average spectrum across segments
power_avg = np.mean(power_spectra, axis=0)
amplitude_avg = np.sqrt(power_avg)

# Remove zero frequency
freqs_nozero = freqs[1:]
amplitude_nozero = amplitude_avg[1:]

# Restrict search band
band_mask = (freqs_nozero >= F_MIN) & (freqs_nozero <= F_MAX)
freqs_band = freqs_nozero[band_mask]
amplitude_band = amplitude_nozero[band_mask]

if len(freqs_band) == 0:
    raise ValueError("No frequencies in selected band. Check F_MIN/F_MAX.")

dominant_index = np.argmax(amplitude_band)
f_shedding = freqs_band[dominant_index]

# Strouhal number*
St = f_shedding * D / U_mean


### Plots and data

In [ ]:
print(f"File: {FILE_PATH.name}")
print(f"Total samples after transient removal: {n_total}")
print(f"Number of segments: {N_SEGMENTS}")
print(f"Samples per segment: {segment_length}")
print(f"Mean velocity U = {U_mean:.4f} m/s")
print(f"Dominant shedding frequency f = {f_shedding:.4f} Hz")
print(f"Strouhal number St = {St:.4f}")
print(f"Drag coefficient, C_D mean = {results_t['C_D'].mean():.4f}, std = {results_t['C_D'].std():.4f}")

# Plot C_L(t)
plt.figure(figsize=(10, 5))
plt.plot(results_t["time"], results_t["C_L"], color="orange")
plt.xlabel("Time [s]")
plt.ylabel("C_L")
plt.title("C_L(t)")
plt.grid(True)

# Plot averaged FFT spectrum
plt.figure(figsize=(10, 5))
plt.plot(freqs_nozero, amplitude_nozero, color="orange", label="Averaged segmented FFT")
plt.axvline(f_shedding, linestyle="dotted", color="indigo", label=f"Peak = {f_shedding:.3f} Hz")
plt.xlim(0, min(200, freqs_nozero.max()))
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.title(f"Averaged FFT of C_L(t) using {N_SEGMENTS} segments")
plt.grid(True)
plt.legend()

plt.show()

# RMS and Variance

### Grouping code (Same as initial grouping code)

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "cylinder": parts[0],
        "re": parts[1],
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = parts[3]
        meta["angle"] = parts[4].replace("A", "")

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

grouped_results = defaultdict(list)

for file in DATA_FOLDER.glob("*.lvm"):
    FILE_PATH = file
    meta = parse_filename(FILE_PATH)

    df = read_lvm_file(FILE_PATH)

    # -----------------------------
    # Raw pressure samples
    # -----------------------------
    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)   # (n_samples, 30)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)   # (n_samples, 30)

    theta1 = np.arange(0.0, 360.0, 12.0)
    theta2 = theta1 + 6.0
    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    # -----------------------------
    # Mean q_inf for normalization
    # -----------------------------
    q_inf_mean = df.iloc[:, 63].mean()

    # -----------------------------
    # Instantaneous Cp samples
    # -----------------------------
    cp1_samples = layer1 / q_inf_mean
    cp2_samples = layer2 / q_inf_mean

    # Mean Cp at each tap
    cp1_mean = cp1_samples.mean(axis=0)
    cp2_mean = cp2_samples.mean(axis=0)

    # Variance and RMS at each tap
    cp1_var = cp1_samples.var(axis=0)
    cp2_var = cp2_samples.var(axis=0)

    cp1_rms = np.sqrt(cp1_var)
    cp2_rms = np.sqrt(cp2_var)

    # -----------------------------
    # Collapse to 6-degree spacing
    # -----------------------------
    cp_collapsed = np.empty(theta_collapsed.size)
    cp_var_collapsed = np.empty(theta_collapsed.size)
    cp_rms_collapsed = np.empty(theta_collapsed.size)

    cp_samples_collapsed = np.empty((cp1_samples.shape[0], theta_collapsed.size))

    # keep your working ordering
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    cp_var_collapsed[0::2] = cp2_var
    cp_var_collapsed[1::2] = cp1_var

    cp_rms_collapsed[0::2] = cp2_rms
    cp_rms_collapsed[1::2] = cp1_rms

    cp_samples_collapsed[:, 0::2] = cp2_samples
    cp_samples_collapsed[:, 1::2] = cp1_samples

    # close loop for plotting/integration
    theta_closed = np.r_[theta_collapsed, 360.0]
    cp_closed = np.r_[cp_collapsed, cp_collapsed[0]]
    cp_var_closed = np.r_[cp_var_collapsed, cp_var_collapsed[0]]
    cp_rms_closed = np.r_[cp_rms_collapsed, cp_rms_collapsed[0]]

    cp_samples_closed = np.column_stack([cp_samples_collapsed, cp_samples_collapsed[:, 0]])

    # -----------------------------
    # Instantaneous CD and CL
    # -----------------------------
    theta_rad = np.deg2rad(theta_closed)

    cd_time = 0.5 * np.trapz(
        cp_samples_closed * np.cos(theta_rad),
        theta_rad,
        axis=1
    )

    cl_time = -0.5 * np.trapz(
        cp_samples_closed * np.sin(theta_rad),
        theta_rad,
        axis=1
    )

    # -----------------------------
    # Statistics for CD and CL
    # -----------------------------
    cd_mean = np.mean(cd_time)
    cl_mean = np.mean(cl_time)

    cd_var = np.var(cd_time)
    cl_var = np.var(cl_time)

    cd_rms = np.std(cd_time)   # RMS of fluctuations
    cl_rms = np.std(cl_time)   # RMS of fluctuations

    # -----------------------------
    # Group key
    # -----------------------------
    if meta["case"] == "Clean":
        group_key = ("Clean",)
    else:
        group_key = ("Trip", meta["trip_mm"], meta["angle"])

    grouped_results[group_key].append({
        "file_name": FILE_PATH.name,
        "re": meta["re"],
        "theta": theta_closed,
        "cp": cp_closed,
        "cp_var": cp_var_closed,
        "cp_rms": cp_rms_closed,

        "cd_time": cd_time,
        "cl_time": cl_time,

        "cd_mean": cd_mean,
        "cl_mean": cl_mean,
        "cd_var": cd_var,
        "cl_var": cl_var,
        "cd_rms": cd_rms,
        "cl_rms": cl_rms
    })

## RMS and variance for C_P

### Variance and RMS for the clean cylinder run

In [ ]:
target_key = ("Clean",)

if target_key in grouped_results:

    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(grouped_results[target_key], key=re_value):

        fig, ax1 = plt.subplots(figsize=(9, 6))
        ax2 = ax1.twinx()

        # --- Cp on left axis ---
        line1, = ax1.plot(
            run_data["theta"],
            run_data["cp"],
            "o-",
            color="crimson",
            linewidth=1.8,
            markersize=5,
            label=r"$C_p$",
        )

        # --- RMS on right axis ---
        line2, = ax2.plot(
            run_data["theta"],
            run_data["cp_rms"],
            "o-",
            color="green",
            linewidth=1.6,
            markersize=4.5,
            label=r"RMS($C_p'$)",
        )

        # --- Variance on right axis ---
        line3, = ax2.plot(
            run_data["theta"],
            run_data["cp_var"],
            "+-",
            color="green",
            linewidth=1.4,
            markersize=6,
            label=r"Var($C_p'$)",
        )

        # Labels and title
        ax1.set_xlabel(r"$\theta$ [deg]", fontsize=16)
        ax1.set_ylabel(r"$C_p$ [-]", color="crimson", fontsize=16)
        ax2.set_ylabel(r"RMS($C_p'$), Var($C_p'$) [-]", color="green", fontsize=16)

        ax1.set_title(run_data["file_name"], fontsize=15)

        ax1.set_xlim(0, 360)
        ax1.set_ylim(-2.5, 1.2)
        ax2.set_ylim(0, 0.5)

        # Axis styling  
        ax1.tick_params(axis="y", colors="crimson", labelsize=11)
        ax2.tick_params(axis="y", colors="green", labelsize=11)
        ax1.tick_params(axis="x", labelsize=11)
        ax1.set_xlim(0, 360)
        ax1.grid(True, alpha=0.3)

        # Combined legend
        lines = [line1, line2, line3]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc="upper right", fontsize=11, framealpha=1)

        plt.tight_layout()

    plt.show()

else:
    print(f"No files found for group {target_key}")
    plt.show()

### Variance and RMS for a specific run

In [ ]:
target_key = ("Trip", "0.94", "27")

if target_key in grouped_results:

    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(grouped_results[target_key], key=re_value):

        fig, ax1 = plt.subplots(figsize=(9, 6))
        ax2 = ax1.twinx()

        # --- Cp on left axis ---
        line1, = ax1.plot(
            run_data["theta"],
            run_data["cp"],
            "o-",
            color="crimson",
            linewidth=1.8,
            markersize=5,
            label=r"$C_p$"
        )

        # --- RMS on right axis ---
        line2, = ax2.plot(
            run_data["theta"],
            run_data["cp_rms"],
            "o-",
            color="green",
            linewidth=1.6,
            markersize=4.5,
            label=r"RMS($C_p'$)"
        )

        # --- Variance on right axis ---
        line3, = ax2.plot(
            run_data["theta"],
            run_data["cp_var"],
            "+-",
            color="green",
            linewidth=1.4,
            markersize=6,
            label=r"Var($C_p'$)"
        )

        # Labels and title
        ax1.set_xlabel(r"$\theta$ [deg]", fontsize=16)
        ax1.set_ylabel(r"$C_p$ [-]", color="crimson", fontsize=16)
        ax2.set_ylabel(r"RMS($C_p'$), Var($C_p'$) [-]", color="green", fontsize=16)

        ax1.set_title(run_data["file_name"], fontsize=15)

        
        ax1.set_xlim(0, 360)
        ax1.set_ylim(-2.5, 1.5)
        ax2.set_ylim(0, 0.4)

        # Axis styling
        ax1.tick_params(axis="y", colors="crimson", labelsize=11)
        ax2.tick_params(axis="y", colors="green", labelsize=11)
        ax1.tick_params(axis="x", labelsize=11)

        ax1.set_xlim(0, 360)
        ax1.grid(True, alpha=0.3)

        # Combined legend
        lines = [line1, line2, line3]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc="upper right", fontsize=11, framealpha=1)

        plt.tight_layout()

    plt.show()

else:
    print(f"No files found for group {target_key}")

## RMS and variance for drag and lift

### Summary

In [ ]:
target_key = ("Clean",)

if target_key in grouped_results:

    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(grouped_results[target_key], key=re_value):
        print(
            f'{run_data["file_name"]}: '
            f'CD_mean = {run_data["cd_mean"]:.4f}, '
            f'CD_rms = {run_data["cd_rms"]:.4f}, '
            f'CD_var = {run_data["cd_var"]:.4f}, '
            f'CL_mean = {run_data["cl_mean"]:.4f}, '
            f'CL_rms = {run_data["cl_rms"]:.4f}, '
            f'CL_var = {run_data["cl_var"]:.4f}'
        )

### Visualization

In [ ]:
target_key = ("Clean",)

if target_key in grouped_results:

    def re_value(run_dict):
        return int(run_dict["re"].replace("Re", "").replace("k", ""))

    for run_data in sorted(grouped_results[target_key], key=re_value):

        fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

        t_cd = np.arange(len(run_data["cd_time"]))
        t_cl = np.arange(len(run_data["cl_time"]))

        # CD(t)
        axes[0].plot(t_cd, run_data["cd_time"], color="tab:blue", linewidth=1.2)
        axes[0].axhline(run_data["cd_mean"], color="black", linestyle="--", linewidth=1.2,
                        label=fr'Mean $C_D$ = {run_data["cd_mean"]:.3f}')
        axes[0].set_ylabel(r"$C_D$")
        axes[0].set_title(run_data["file_name"])
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        # CL(t)
        axes[1].plot(t_cl, run_data["cl_time"], color="tab:red", linewidth=1.2)
        axes[1].axhline(run_data["cl_mean"], color="black", linestyle="--", linewidth=1.2,
                        label=fr'Mean $C_L$ = {run_data["cl_mean"]:.3f}')
        axes[1].set_xlabel("Sample index")
        axes[1].set_ylabel(r"$C_L$")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        plt.tight_layout()

    plt.show()

else:
    print(f"No files found for group {target_key}")

# Comparisons

## C_D for different runs

### Target Key 

In [ ]:
selected_case = "Trip"
selected_trip = 0.94
selected_angle = 69


selected_case1 = "Trip"
selected_trip1 = 0.15
selected_angle1 = 69

### Code

In [ ]:


DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")

def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()

def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "file": file_path.name,
        "re": int(parts[1].replace("Re", "").replace("k", "")) * 1000,
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = float(parts[3])
        meta["angle"] = int(parts[4].replace("A", ""))

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta

results = []

for FILE_PATH in sorted(DATA_FOLDER.glob("*.lvm")):
    meta = parse_filename(FILE_PATH)
    df = read_lvm_file(FILE_PATH)

    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    p1_mean = layer1.mean(axis=0)
    p2_mean = layer2.mean(axis=0)

    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    q_inf_mean = df.iloc[:, 63].mean()   # pitot channel

    cp1_mean = p1_mean / q_inf_mean
    cp2_mean = p2_mean / q_inf_mean

    cp_collapsed = np.empty(theta_collapsed.size)
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    cp_collapsed_closed = np.r_[cp_collapsed, cp_collapsed[0]]

    theta_rad_closed = np.deg2rad(theta_collapsed_closed)

    # positive sign to match your current angle convention
    C_D = 0.5 * np.trapz(cp_collapsed_closed * np.cos(theta_rad_closed), theta_rad_closed)
    C_L = 0.5 * np.trapz(cp_collapsed_closed * np.sin(theta_rad_closed), theta_rad_closed)

    results.append({
        "file": meta["file"],
        "Re": meta["re"],
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "C_D": C_D,
        "C_L": C_L
    })

results_df = pd.DataFrame(results)

# -----------------------------
# Build configuration labels
# -----------------------------
def config_label(row):
    if row["case"] == "Clean":
        return "Clean"
    return f'Trip {row["trip_mm"]:.2f} mm A{int(row["angle"])}'

results_df["config"] = results_df.apply(config_label, axis=1)

### Plots

In [ ]:
run_df = results_df[
    (results_df["case"] == selected_case) &
    (results_df["trip_mm"] == selected_trip) &
    (results_df["angle"] == selected_angle)
].sort_values("Re")

if run_df.empty:
    print("No matching run found.")
else:
    plt.figure(figsize=(9, 5))
    plt.plot(run_df["Re"], run_df["C_D"], "o-", label=f"A{int(selected_angle)}")

    plt.xlabel("Reynolds number")
    plt.ylabel("$C_D$")
    plt.ylim(0.2, 1.3)
    plt.title(f"$C_D$ vs Reynolds number - {selected_case}, {selected_trip:.2f} mm")
    plt.grid(True)
    plt.legend()
    plt.show()
    
run_df = results_df[
    (results_df["case"] == selected_case1) &
    (results_df["trip_mm"] == selected_trip1) &
    (results_df["angle"] == selected_angle1)
].sort_values("Re")

if run_df.empty:
    print("No matching run found.")
else:
    plt.figure(figsize=(9, 5))
    plt.plot(run_df["Re"], run_df["C_D"], "o-", label=f"A{int(selected_angle1)}")

    plt.xlabel("Reynolds number")
    plt.ylabel("$C_D$")
    plt.ylim(0.2, 1.3)
    plt.title(f"$C_D$ vs Reynolds number - {selected_case1}, {selected_trip1:.2f} mm")
    plt.grid(True)
    plt.legend()
    plt.show()



## Cylinder Plots

### Target keys

In [ ]:
target_key_1 = ("Clean",)
target_re_1 = "Re100k"

target_key_2 = ("Trip", "0.94", "63")
target_re_2 = "Re100k"

### Code

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Only needed if run_data does NOT already contain a full file path
DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

D = 0.100   # cylinder diameter [m]

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)
EXTRA_NAMES = ["pitot", "AoA", "F", "alpha", "P_m", "V_tunnel", "rho", "time"]

#Max Scale
CP_ABS_MAX_FIXED = 2.0

# =========================
# HELPERS
# =========================
def re_value(run_dict):
    return int(run_dict["re"].replace("Re", "").replace("k", ""))


def get_filtered_run(grouped_results, target_key, target_re):
    if target_key not in grouped_results:
        raise ValueError(f"No files found for group {target_key}")

    runs = sorted(grouped_results[target_key], key=re_value)

    if target_re is not None:
        runs = [run for run in runs if run["re"] == target_re]

    if not runs:
        raise ValueError(f"No files found for group {target_key} with Reynolds number {target_re}")

    if len(runs) > 1:
        raise ValueError(
            f"More than one run matched {target_key} and {target_re}. "
            f"Expected exactly one."
        )

    return runs[0]


def resolve_file_path(run_data, data_folder=None):
    """
    Tries to get the raw .lvm file path for a run.
    Priority:
    1) run_data['file_path']
    2) data_folder / run_data['file_name']
    """
    if "file_path" in run_data:
        return Path(run_data["file_path"])

    if data_folder is not None and "file_name" in run_data:
        return data_folder / run_data["file_name"]

    raise ValueError(
        f"Could not resolve file path for {run_data.get('file_name', 'unknown file')}. "
        f"Add run_data['file_path'] or set DATA_FOLDER correctly."
    )


def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")


def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()


def compute_cd_from_mean_cp(run_data):
    """
    Computes C_D from the mean C_p(theta) already stored in run_data.
    Useful as fallback if raw time-series file is unavailable.
    """
    theta_deg = np.array(run_data["theta"])
    cp_vals = np.array(run_data["cp"])

    theta_rad = np.deg2rad(theta_deg)

    # Same sign convention as your earlier code
    cd = 0.5 * np.trapz(cp_vals * np.cos(theta_rad), theta_rad)
    return cd


def compute_cd_st_from_lvm(file_path, D=0.100, transient_fraction=0.10, f_min=1.0, f_max=500.0):
    """
    Reads the raw .lvm file and computes:
    - mean C_D from time-resolved pressure signals
    - Strouhal number from FFT of C_L(t)
    """
    df = read_lvm_file(file_path)

    # Pressure tap data
    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)   # shape (n_samples, 30)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)   # shape (n_samples, 30)

    # Extra channels
    extra_data = df.iloc[:, 63:71].copy()
    extra_data.columns = EXTRA_NAMES

    time = extra_data["time"].to_numpy()
    q_inf = extra_data["pitot"].to_numpy()
    U_tunnel = extra_data["V_tunnel"].to_numpy()

    # Avoid division by zero
    eps = 1e-12
    q_inf = np.where(np.abs(q_inf) < eps, np.nan, q_inf)

    # Cp for every sample
    cp1 = layer1 / q_inf[:, None]
    cp2 = layer2 / q_inf[:, None]

    # Angular positions
    theta_collapsed = np.arange(0.0, 360.0, 6.0)   # 60 positions
    theta_collapsed_closed = np.r_[theta_collapsed, 360.0]
    theta_rad_closed = np.deg2rad(theta_collapsed_closed)

    # Interleave tap rows into one circumferential Cp distribution per sample
    n_samples = df.shape[0]
    cp_collapsed = np.empty((n_samples, theta_collapsed.size))
    cp_collapsed[:, 0::2] = cp2
    cp_collapsed[:, 1::2] = cp1

    # Close the loop for integration
    cp_collapsed_closed = np.hstack([cp_collapsed, cp_collapsed[:, [0]]])

    # Compute C_L(t) and C_D(t)
    C_L_t = 0.5 * np.trapz(
        cp_collapsed_closed * np.sin(theta_rad_closed),
        theta_rad_closed,
        axis=1
    )
    C_D_t = 0.5 * np.trapz(
        cp_collapsed_closed * np.cos(theta_rad_closed),
        theta_rad_closed,
        axis=1
    )

    results_t = pd.DataFrame({
        "time": time,
        "C_L": C_L_t,
        "C_D": C_D_t,
        "V_tunnel": U_tunnel
    })

    results_t = results_t.replace([np.inf, -np.inf], np.nan).dropna(subset=["time", "C_L", "C_D", "V_tunnel"])

    if len(results_t) < 10:
        raise ValueError(f"Too few valid samples after cleaning in file: {file_path.name}")

    # Remove initial transient
    start_idx = int(transient_fraction * len(results_t))
    results_fft = results_t.iloc[start_idx:].copy()

    time_fft = results_fft["time"].to_numpy()
    cl_fft = results_fft["C_L"].to_numpy()

    if len(time_fft) < 10:
        raise ValueError(f"Too few samples left for FFT in file: {file_path.name}")

    # Remove mean from C_L(t)
    cl_fluct = cl_fft - np.mean(cl_fft)

    # Sampling interval
    dt = np.mean(np.diff(time_fft))

    # FFT with Hann window
    n = len(cl_fluct)
    window = np.hanning(n)
    cl_windowed = cl_fluct * window

    freqs = np.fft.rfftfreq(n, d=dt)
    fft_vals = np.fft.rfft(cl_windowed)
    amplitude = np.abs(fft_vals)

    # Remove zero-frequency component
    freqs_nozero = freqs[1:]
    amplitude_nozero = amplitude[1:]

    band_mask = (freqs_nozero >= f_min) & (freqs_nozero <= f_max)

    freqs_band = freqs_nozero[band_mask]
    amplitude_band = amplitude_nozero[band_mask]

    if len(freqs_band) == 0:
        raise ValueError(
            f"No frequency content found in the selected band [{f_min}, {f_max}] Hz "
            f"for file: {file_path.name}"
        )

    dominant_index = np.argmax(amplitude_band)
    f_shedding = freqs_band[dominant_index]

    U_mean = results_fft["V_tunnel"].mean()
    St = f_shedding * D / U_mean

    cd_mean = results_t["C_D"].mean()

    return cd_mean, St


def get_cd_st_for_run(run_data, data_folder=None, D=0.100):
    """
    Preferred:
    - compute C_D mean and St from raw .lvm file

    Fallback:
    - compute only C_D from mean C_p(theta), set St = None
    """
    try:
        file_path = resolve_file_path(run_data, data_folder=data_folder)
        cd_mean, st = compute_cd_st_from_lvm(file_path, D=D)
        return cd_mean, st
    except Exception as e:
        print(f"Warning: Could not compute St from raw file for {run_data.get('file_name', 'unknown file')}")
        print(f"Reason: {e}")
        print("Falling back to C_D from mean C_p only.")
        cd_mean = compute_cd_from_mean_cp(run_data)
        return cd_mean, None


def draw_cp_normal_bars(ax, run_data, cp_abs_max, cd_value=None, st_value=None,
                        radius=1.0, max_bar_length=0.35):
    theta_deg = np.array(run_data["theta"][:-1])   # drop repeated 360° point
    cp_vals = np.array(run_data["cp"][:-1])
    theta_rad = np.deg2rad(theta_deg)

    scale = max_bar_length / cp_abs_max if cp_abs_max != 0 else 1.0

    # Cylinder surface coordinates
    x = -radius * np.cos(theta_rad)
    y =  radius * np.sin(theta_rad)

    # Outward unit normals
    nx = -np.cos(theta_rad)
    ny =  np.sin(theta_rad)

    # Cylinder outline
    phi = np.linspace(0, 2*np.pi, 400)
    ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.5)

    # Draw Cp bars
    for xs, ys, nxi, nyi, cp in zip(x, y, nx, ny, cp_vals):
        bar_len = scale * abs(cp)
        xe = xs + bar_len * nxi
        ye = ys + bar_len * nyi

        color = "crimson" if cp >= 0 else "royalblue"
        ax.plot([xs, xe], [ys, ye], color=color, lw=2.5, solid_capstyle="round")
        ax.plot(xs, ys, "ko", ms=3)

    # Angle labels every 30 degrees

    x = -x
    y = -y
    for xs, ys, ang in zip(x, y, theta_deg):
        if ang % 30 == 0:
            ax.text(
                -0.9 * xs,
                -0.9 * ys,
                f"{int(ang)}°",
                fontsize=8,
                ha="center",
                va="center"
            )

    # Metrics box
    if cd_value is not None and st_value is not None:
        metrics_text = f"$C_D$ = {cd_value:.3f}\n$St$ = {st_value:.3f}"
    elif cd_value is not None:
        metrics_text = f"$C_D$ = {cd_value:.3f}\n$St$ = N/A"
    else:
        metrics_text = "$C_D$ = N/A\n$St$ = N/A"

    ax.text(
        0.03, 0.97,
        metrics_text,
        transform=ax.transAxes,
        fontsize=10,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
    )

    ax.set_aspect("equal")
    ax.set_xlim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_ylim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True)


# =========================
# GET SELECTED RUNS
# =========================
run1 = get_filtered_run(grouped_results, target_key_1, target_re_1)
run2 = get_filtered_run(grouped_results, target_key_2, target_re_2)

# Compute C_D and St for each run
cd1, st1 = get_cd_st_for_run(run1, data_folder=DATA_FOLDER, D=D)
cd2, st2 = get_cd_st_for_run(run2, data_folder=DATA_FOLDER, D=D)

# Fixed Cp scale so plots are comparable across files
cp_abs_max = CP_ABS_MAX_FIXED


### Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

draw_cp_normal_bars(axes[0], run1, cp_abs_max, cd_value=cd1, st_value=st1)
draw_cp_normal_bars(axes[1], run2, cp_abs_max, cd_value=cd2, st_value=st2)

axes[0].set_title(f'{run1["file_name"]} - Day 1')
axes[1].set_title(f'{run2["file_name"]} - Day 1')


plt.tight_layout()
plt.show()

## Cylinder with tripwire

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Example
# =========================
target_key_1 = ("Trip", "0.94", "33")
target_re_1 = "Re100k"

target_key_2 = ("Trip", "0.94", "27")
target_re_2 = "Re100k"

def re_value(run_dict):
    return int(run_dict["re"].replace("Re", "").replace("k", ""))

def get_trip_angles_from_key(target_key):
    """
    Read trip-wire angles directly from target_key.
    Expected format for trip case: ("Trip", wire_diameter, angle)
    """
    if target_key[0] != "Trip":
        return []

    trip_angle = float(target_key[2])
    return [trip_angle, 360.0 - trip_angle]

def draw_trip_wire_arc(ax, center_angle_deg, radius=1.0,
                       arc_halfwidth_deg=2.5,
                       arc_radius_scale=1.015,
                       color="darkorange",
                       lw=4,
                       show_center_dot=True):
    arc_angles_deg = np.linspace(
        center_angle_deg - arc_halfwidth_deg,
        center_angle_deg + arc_halfwidth_deg,
        80
    )
    arc_angles_rad = np.deg2rad(arc_angles_deg)

    r_arc = radius * arc_radius_scale

    x_arc = -r_arc * np.cos(arc_angles_rad)
    y_arc =  r_arc * np.sin(arc_angles_rad)

    ax.plot(
        x_arc, y_arc,
        color=color,
        lw=lw,
        solid_capstyle="round",
        zorder=6
    )

    if show_center_dot:
        ang_rad = np.deg2rad(center_angle_deg)
        xt = -r_arc * np.cos(ang_rad)
        yt =  r_arc * np.sin(ang_rad)

        ax.plot(
            xt, yt,
            marker="o",
            ms=6,
            mec="black",
            mfc="gold",
            zorder=7
        )

def draw_cp_normal_bars(ax, run_data, cp_abs_max, trip_angles=None,
                        radius=1.0, max_bar_length=0.35,
                        show_trip_arc=True, trip_arc_halfwidth_deg=0.5):
    theta_deg = np.array(run_data["theta"][:-1])   # drop repeated 360° point
    cp_vals = np.array(run_data["cp"][:-1])
    theta_rad = np.deg2rad(theta_deg)

    scale = max_bar_length / cp_abs_max if cp_abs_max != 0 else 1.0

    # Cylinder surface coordinates
    x = -radius * np.cos(theta_rad)
    y =  radius * np.sin(theta_rad)

    # Outward unit normals
    nx = -np.cos(theta_rad)
    ny =  np.sin(theta_rad)

    # Cylinder outline
    phi = np.linspace(0, 2*np.pi, 400)
    ax.plot(-radius*np.cos(phi), radius*np.sin(phi), "k-", lw=1.5)

    # Draw Cp bars
    for xs, ys, nxi, nyi, cp in zip(x, y, nx, ny, cp_vals):
        bar_len = scale * abs(cp)
        xe = xs + bar_len * nxi
        ye = ys + bar_len * nyi

        color = "crimson" if cp >= 0 else "royalblue"
        ax.plot([xs, xe], [ys, ye], color=color, lw=2.5, solid_capstyle="round")
        ax.plot(xs, ys, "ko", ms=3)

    # Plot trip-wire arcs
    if show_trip_arc and trip_angles:
        for ang in trip_angles:
            draw_trip_wire_arc(
                ax,
                center_angle_deg=ang,
                radius=radius,
                arc_halfwidth_deg=trip_arc_halfwidth_deg,
                arc_radius_scale=1.015,
                color="darkorange",
                lw=4,
                show_center_dot=True
            )

    # Angle labels every 30 degrees
    
    x = -x
    y = -y
    for xs, ys, ang in zip(x, y, theta_deg):
        if ang % 30 == 0:
            ax.text(
                -0.9 * xs,
                -0.9 * ys,
                f"{int(ang)}°",
                fontsize=8,
                ha="center",
                va="center"
            )

    ax.set_aspect("equal")
    ax.set_xlim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_ylim(-(radius + max_bar_length + 0.2), radius + max_bar_length + 0.2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True)
    ax.set_title(f'{run_data["file_name"]}')

def get_filtered_run(grouped_results, target_key, target_re):
    if target_key not in grouped_results:
        raise ValueError(f"No files found for group {target_key}")

    runs = sorted(grouped_results[target_key], key=re_value)

    if target_re is not None:
        runs = [run for run in runs if run["re"] == target_re]

    if not runs:
        raise ValueError(f"No files found for group {target_key} with Reynolds number {target_re}")

    if len(runs) > 1:
        raise ValueError(
            f"More than one run matched {target_key} and {target_re}. "
            f"Expected exactly one."
        )

    return runs[0]


# Get the two selected runs
run1 = get_filtered_run(grouped_results, target_key_1, target_re_1)
run2 = get_filtered_run(grouped_results, target_key_2, target_re_2)

# Trip angles taken directly from the keys
trip_angles_1 = get_trip_angles_from_key(target_key_1)
trip_angles_2 = get_trip_angles_from_key(target_key_2)

# Use common Cp scale across both plots
cp_abs_max = max(
    np.max(np.abs(run1["cp"][:-1])),
    np.max(np.abs(run2["cp"][:-1]))
)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

draw_cp_normal_bars(axes[0], run1, cp_abs_max, trip_angles=trip_angles_1)
draw_cp_normal_bars(axes[1], run2, cp_abs_max, trip_angles=trip_angles_2)

plt.tight_layout()


# Error

## Initial attempt

### Grouping

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

N_BATCHES = 5   # number of batches used for uncertainty/error bars


def read_lvm_file(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")


def extract_1based_inclusive(df: pd.DataFrame, start: int, end: int) -> np.ndarray:
    return df.iloc[:, start - 1:end].to_numpy()


def parse_filename(file_path: Path):
    parts = file_path.stem.split("_")

    meta = {
        "file": file_path.name,
        "re": int(parts[1].replace("Re", "").replace("k", "")) * 1000,
        "case": None,
        "trip_mm": None,
        "angle": None
    }

    if len(parts) == 3 and parts[2] == "Clean":
        meta["case"] = "Clean"

    elif len(parts) == 5 and parts[2] == "Trip":
        meta["case"] = "Trip"
        meta["trip_mm"] = float(parts[3])
        meta["angle"] = int(parts[4].replace("A", ""))

    else:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return meta


def t_critical_95(n_batches):
    """
    Two-sided 95% t-critical values.
    For N_BATCHES = 5, df = 4, so t = 2.776.
    """

    t_values = {
        2: 12.706,
        3: 4.303,
        4: 3.182,
        5: 2.776,
        6: 2.571,
        7: 2.447,
        8: 2.365,
        9: 2.306,
        10: 2.262,
    }

    if n_batches in t_values:
        return t_values[n_batches]
    else:
        return 1.96   # approximate value for larger number of batches


def compute_cd_cl(theta_deg, cp_vals):

    theta_closed = np.r_[theta_deg, 360.0]
    cp_closed = np.r_[cp_vals, cp_vals[0]]

    theta_rad_closed = np.deg2rad(theta_closed)

    C_D = 0.5 * np.trapz(cp_closed * np.cos(theta_rad_closed), theta_rad_closed)
    C_L = 0.5 * np.trapz(cp_closed * np.sin(theta_rad_closed), theta_rad_closed)

    return C_D, C_L


def compute_batch_cd_cl(layer1, layer2, q_inf, theta_collapsed, n_batches=5):

    #Splits the time signal into batches and computes C_D and C_L for each batch.

    #This gives an estimate of the finite-sampling uncertainty.
    

    layer1_batches = np.array_split(layer1, n_batches, axis=0)
    layer2_batches = np.array_split(layer2, n_batches, axis=0)
    q_inf_batches = np.array_split(q_inf, n_batches, axis=0)

    C_D_batches = []
    C_L_batches = []

    for l1_batch, l2_batch, q_batch in zip(layer1_batches, layer2_batches, q_inf_batches):

        p1_mean_batch = l1_batch.mean(axis=0)
        p2_mean_batch = l2_batch.mean(axis=0)

        q_inf_mean_batch = q_batch.mean()

        cp1_mean_batch = p1_mean_batch / q_inf_mean_batch
        cp2_mean_batch = p2_mean_batch / q_inf_mean_batch

        cp_collapsed_batch = np.empty(theta_collapsed.size)
        cp_collapsed_batch[0::2] = cp2_mean_batch
        cp_collapsed_batch[1::2] = cp1_mean_batch

        C_D_batch, C_L_batch = compute_cd_cl(theta_collapsed, cp_collapsed_batch)

        C_D_batches.append(C_D_batch)
        C_L_batches.append(C_L_batch)

    C_D_batches = np.array(C_D_batches)
    C_L_batches = np.array(C_L_batches)

    C_D_mean = C_D_batches.mean()
    C_L_mean = C_L_batches.mean()

    C_D_std = C_D_batches.std(ddof=1)
    C_L_std = C_L_batches.std(ddof=1)

    tcrit = t_critical_95(n_batches)

    C_D_ci95 = tcrit * C_D_std / np.sqrt(n_batches)
    C_L_ci95 = tcrit * C_L_std / np.sqrt(n_batches)

    return C_D_mean, C_L_mean, C_D_ci95, C_L_ci95, C_D_batches, C_L_batches


results = []

for FILE_PATH in sorted(DATA_FOLDER.glob("*.lvm")):
    meta = parse_filename(FILE_PATH)
    df = read_lvm_file(FILE_PATH)

    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    theta_collapsed = np.arange(0.0, 360.0, 6.0)

    # Pitot/dynamic pressure channel
    q_inf = df.iloc[:, 63].to_numpy()

    # -----------------------------
    # Original full-signal calculation
    # -----------------------------
    p1_mean = layer1.mean(axis=0)
    p2_mean = layer2.mean(axis=0)

    q_inf_mean = q_inf.mean()

    cp1_mean = p1_mean / q_inf_mean
    cp2_mean = p2_mean / q_inf_mean

    cp_collapsed = np.empty(theta_collapsed.size)
    cp_collapsed[0::2] = cp2_mean
    cp_collapsed[1::2] = cp1_mean

    C_D, C_L = compute_cd_cl(theta_collapsed, cp_collapsed)

    # -----------------------------
    # Batch-based uncertainty
    # -----------------------------
    C_D_batch_mean, C_L_batch_mean, C_D_ci95, C_L_ci95, C_D_batches, C_L_batches = compute_batch_cd_cl(
        layer1=layer1,
        layer2=layer2,
        q_inf=q_inf,
        theta_collapsed=theta_collapsed,
        n_batches=N_BATCHES
    )

    results.append({
        "file": meta["file"],
        "Re": meta["re"],
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],

        # Main values
        "C_D": C_D,
        "C_L": C_L,

        # Error bars: 95% confidence interval
        "C_D_ci95": C_D_ci95,
        "C_L_ci95": C_L_ci95,

        # Optional diagnostic values
        "C_D_batch_mean": C_D_batch_mean,
        "C_L_batch_mean": C_L_batch_mean,
        "C_D_batches": C_D_batches,
        "C_L_batches": C_L_batches,
    })

results_df = pd.DataFrame(results)


# -----------------------------
# Build configuration labels
# -----------------------------
def config_label(row):
    if row["case"] == "Clean":
        return "Clean"
    return f'Trip {row["trip_mm"]:.2f} mm A{int(row["angle"])}'


results_df["config"] = results_df.apply(config_label, axis=1)

In [ ]:
print("Batch sizes:", [len(batch) for batch in layer1_batches])

In [ ]:
plt.figure(figsize=(9, 6))

for config, group in results_df.groupby("config"):
    group = group.sort_values("Re")

    plt.errorbar(
        group["Re"],
        group["C_D"],
        yerr=group["C_D_ci95"],
        fmt="o-",
        capsize=4,
        linewidth=1.5,
        markersize=5,
        label=config
    )

plt.xlabel("Reynolds number")
plt.ylabel(r"$C_D$")
plt.title(r"$C_D$ vs Reynolds number with 95% confidence intervals")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

for config, group in results_df.groupby("config"):
    group = group.sort_values("Re")

    plt.errorbar(
        group["Re"],
        group["C_L"],
        yerr=group["C_L_ci95"],
        fmt="o-",
        capsize=4,
        linewidth=1.5,
        markersize=5,
        label=config
    )

plt.xlabel("Reynolds number")
plt.ylabel(r"$C_L$")
plt.title(r"$C_L$ vs Reynolds number with 95% confidence intervals")
plt.grid(True)
plt.legend()
plt.show()

### Deviation plot code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_cd_batch_ridgeline(
    results_df,
    case="Trip",
    trip_mm=0.94,
    angle=27,
    title=None,
    show_values=True
):
    """
    Ridgeline-style visualization of batch-based C_D values.

    Each horizontal distribution represents the batch C_D values for one Reynolds number.
    The orange point shows the full-signal mean C_D.
    The horizontal black line shows the 95% confidence interval.
    """





    # -----------------------------
    # Filter data
    # -----------------------------
    df_plot = results_df[results_df["case"] == case].copy()

    if case == "Trip":
        if trip_mm is not None:
            df_plot = df_plot[df_plot["trip_mm"] == trip_mm]

        if angle is not None:
            df_plot = df_plot[df_plot["angle"] == angle]

    df_plot = df_plot.sort_values("Re")

    if df_plot.empty:
        raise ValueError("No data found for the selected configuration.")





    # -----------------------------
    # Prepare data
    # -----------------------------


    batch_data = [np.asarray(row["C_D_batches"], dtype=float) for _, row in df_plot.iterrows()]
    reynolds_labels = [f"{int(row['Re']/1000)}k" for _, row in df_plot.iterrows()]

    y_positions = np.arange(len(df_plot))




    # -----------------------------
    # Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(10, 6))

    parts = ax.violinplot(
        batch_data,
        positions=y_positions,
        vert=False,
        widths=0.75,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    # Style violins
    for body in parts["bodies"]:
        body.set_alpha(0.45)

    # Add batch points, mean points, and confidence intervals
    for i, (_, row) in enumerate(df_plot.iterrows()):
        cd_batches = np.asarray(row["C_D_batches"], dtype=float)
        cd_mean = row["C_D"]
        cd_ci95 = row["C_D_ci95"]

        # Small vertical jitter so the batch points do not overlap perfectly
        jitter = np.linspace(-0.12, 0.12, len(cd_batches))

        ax.scatter(
            cd_batches,
            np.full_like(cd_batches, y_positions[i], dtype=float) + jitter,
            s=35,
            alpha=0.8,
            color="steelblue",
            edgecolor="black",
            linewidth=0.4,
            zorder=3,
            label="Batch values" if i == 0 else None
        )

        # 95% CI line
        ax.hlines(
            y=y_positions[i],
            xmin=cd_mean - cd_ci95,
            xmax=cd_mean + cd_ci95,
            color="black",
            linewidth=2.0,
            zorder=4,
            label="95% CI" if i == 0 else None
        )

        # Mean point
        ax.scatter(
            cd_mean,
            y_positions[i],
            s=70,
            color="darkorange",
            edgecolor="black",
            linewidth=0.5,
            zorder=5,
            label=r"Mean $C_D$" if i == 0 else None
        )

        # Optional text values
        if show_values:
            value_text = rf"$C_D={cd_mean:.3f}\ \pm\ {cd_ci95:.3f}$"

            ax.text(
                cd_mean + cd_ci95 + 0.015,
                y_positions[i],
                value_text,
                va="center",
                fontsize=9
            )




    # -----------------------------
    # Axis formatting
    # -----------------------------


    ax.set_yticks(y_positions)
    ax.set_yticklabels(reynolds_labels)

    ax.set_xlabel(r"$C_D$")
    ax.set_ylabel("Reynolds number")

    if title is None:
        if case == "Clean":
            title = r"Batch-based spread of $C_D$ for clean cylinder"
        else:
            title = rf"Batch-based spread of $C_D$ for trip {trip_mm:.2f} mm at A{angle}"

    ax.set_title(title)

    ax.grid(True, axis="x", alpha=0.35)
    ax.legend(loc="best")

    plt.tight_layout()
    plt.show()

### Plot

In [ ]:
plot_cd_batch_ridgeline(
    results_df,
    case="Trip",
    trip_mm=0.94,angle=27,
)

## Second attempt

### Grouping Code

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# =============================
# User settings
# =============================

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS_1BASED = (1, 30)
LAYER2_COLS_1BASED = (33, 62)

EXTRA_NAMES = ["pitot", "AoA", "F", "alpha", "P_m", "V_tunnel", "rho", "time"]

N_BATCHES = 10


# =============================
# Basic helper functions
# =============================

def read_lvm_file(file_path):
    """Reads one .lvm file."""
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")


def extract_1based_inclusive(df, start, end):
    """Extracts columns using 1-based inclusive indexing."""
    return df.iloc[:, start - 1:end].to_numpy()


def parse_filename(file_path):
    """
    Extracts Reynolds number, case, trip thickness and angle from filename.

    Expected examples:
    Cyl100_Re100k_Clean.lvm
    Cyl100_Re250k_Trip_0.94_A69.lvm
    """

    parts = file_path.stem.split("_")

    re_label = None
    case = None
    trip_mm = None
    angle = None

    for part in parts:
        if part.startswith("Re"):
            re_label = part

    if "Clean" in parts:
        case = "Clean"
        key = ("Clean",)

    elif "Trip" in parts:
        case = "Trip"

        trip_index = parts.index("Trip")
        trip_mm = parts[trip_index + 1]

        for part in parts:
            if part.startswith("A"):
                angle = part.replace("A", "")

        key = ("Trip", trip_mm, angle)

    else:
        print(f"Skipping file because case was not recognized: {file_path.name}")
        return None

    return {
        "re": re_label,
        "case": case,
        "trip_mm": trip_mm,
        "angle": angle,
        "key": key
    }


def re_to_number(re_label):
    """Converts 'Re250k' to 250000."""
    return int(re_label.replace("Re", "").replace("k", "")) * 1000


def collapse_pressure_taps(layer1_values, layer2_values):
    """
    Combines layer 1 and layer 2 pressure taps into one 0-354 degree distribution.
    Layer 1: 0, 12, 24, ..., 348 deg
    Layer 2: 6, 18, 30, ..., 354 deg
    """

    theta_layer1 = np.arange(0, 360, 12)
    theta_layer2 = np.arange(6, 360, 12)

    theta_all = np.r_[theta_layer1, theta_layer2]
    values_all = np.r_[layer1_values, layer2_values]

    sort_idx = np.argsort(theta_all)

    theta_sorted = theta_all[sort_idx]
    values_sorted = values_all[sort_idx]

    return theta_sorted, values_sorted


def compute_cd(theta_deg, cp_vals):
    """
    Computes C_D from pressure coefficient distribution.
    """

    theta_closed = np.r_[theta_deg, 360.0]
    cp_closed = np.r_[cp_vals, cp_vals[0]]

    theta_rad = np.deg2rad(theta_closed)

    C_D = 0.5 * np.trapz(cp_closed * np.cos(theta_rad), theta_rad)

    return C_D


# =============================
# Main grouping code
# =============================

grouped_results = {}

for file_path in sorted(DATA_FOLDER.glob("*.lvm")):

    meta = parse_filename(file_path)

    if meta is None:
        continue

    df = read_lvm_file(file_path)

    # -----------------------------
    # Extract pressure tap data
    # -----------------------------
    layer1 = extract_1based_inclusive(df, *LAYER1_COLS_1BASED)
    layer2 = extract_1based_inclusive(df, *LAYER2_COLS_1BASED)

    # -----------------------------
    # Extract extra columns
    # -----------------------------
    extra_start_col = LAYER2_COLS_1BASED[1]  # zero-based start after column 62
    extra = df.iloc[:, extra_start_col:extra_start_col + len(EXTRA_NAMES)].copy()
    extra.columns = EXTRA_NAMES[:extra.shape[1]]

    q_inf = extra["pitot"].to_numpy()

    # Avoid divide-by-zero issues
    q_inf_mean = np.mean(q_inf)

    # -----------------------------
    # Mean Cp distribution
    # -----------------------------
    layer1_mean = np.mean(layer1, axis=0)
    layer2_mean = np.mean(layer2, axis=0)

    cp_layer1 = layer1_mean / q_inf_mean
    cp_layer2 = layer2_mean / q_inf_mean

    theta, cp = collapse_pressure_taps(cp_layer1, cp_layer2)

    # -----------------------------
    # RMS Cp distribution
    # -----------------------------
    cp_time_layer1 = layer1 / q_inf[:, None]
    cp_time_layer2 = layer2 / q_inf[:, None]

    cp_rms_layer1 = np.std(cp_time_layer1, axis=0, ddof=1)
    cp_rms_layer2 = np.std(cp_time_layer2, axis=0, ddof=1)

    _, cp_rms = collapse_pressure_taps(cp_rms_layer1, cp_rms_layer2)

    # -----------------------------
    # Batch Cp and batch C_D
    # -----------------------------
    layer1_batches = np.array_split(layer1, N_BATCHES, axis=0)
    layer2_batches = np.array_split(layer2, N_BATCHES, axis=0)
    q_batches = np.array_split(q_inf, N_BATCHES, axis=0)

    cp_batches = []
    C_D_batches = []

    for l1_batch, l2_batch, q_batch in zip(layer1_batches, layer2_batches, q_batches):

        q_batch_mean = np.mean(q_batch)

        l1_batch_mean = np.mean(l1_batch, axis=0)
        l2_batch_mean = np.mean(l2_batch, axis=0)

        cp_l1_batch = l1_batch_mean / q_batch_mean
        cp_l2_batch = l2_batch_mean / q_batch_mean

        theta_batch, cp_batch = collapse_pressure_taps(cp_l1_batch, cp_l2_batch)

        C_D_batch = compute_cd(theta_batch, cp_batch)

        cp_batches.append(cp_batch)
        C_D_batches.append(C_D_batch)

    # -----------------------------
    # Store run data
    # -----------------------------
    run_data = {
        "file": file_path.name,
        "re": meta["re"],
        "Re": re_to_number(meta["re"]),
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "theta": theta,
        "cp": cp,
        "cp_rms": cp_rms,
        "cp_batches": cp_batches,
        "C_D": compute_cd(theta, cp),
        "C_D_batches": C_D_batches,
    }

    key = meta["key"]

    if key not in grouped_results:
        grouped_results[key] = []

    grouped_results[key].append(run_data)


# =============================
# Check what was grouped
# =============================

for key, runs in grouped_results.items():
    print(key, ":", len(runs), "runs")

### Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =============================
# User settings
# =============================
TARGET_TRIP_MM = "0.94"
N_BATCHES = 5

# =============================
# Helper functions
# =============================

def re_value(run_dict):
    """Convert 'Re150k' into 150000 for sorting."""
    return int(run_dict["re"].replace("Re", "").replace("k", "")) * 1000


def compute_cd(theta_deg, cp_vals):
    """
    Compute C_D from one pressure distribution.
    Uses the same pressure integration method as before.
    """

    theta_deg = np.asarray(theta_deg)
    cp_vals = np.asarray(cp_vals)

    # Close the curve if it is not already closed
    if theta_deg[-1] != 360:
        theta_deg = np.r_[theta_deg, 360.0]
        cp_vals = np.r_[cp_vals, cp_vals[0]]

    theta_rad = np.deg2rad(theta_deg)

    C_D = 0.5 * np.trapz(cp_vals * np.cos(theta_rad), theta_rad)

    return C_D


def mean_and_error_from_batches(values):
    """
    Calculates mean value and 95% confidence interval error
    from batch values.
    """

    values = np.asarray(values)

    mean_val = np.mean(values)
    std_val = np.std(values, ddof=1)

    k = len(values)

    # For 5 batches: df = 4, 95% t-value = 2.776
    # This is the value we normally use here.
    if k == 5:
        t_val = 2.776
    else:
        # fallback approximation
        t_val = 1.96

    error = t_val * std_val / np.sqrt(k)

    return mean_val, error


# =============================
# Plot C_D with error bars
# for all 0.94 mm tripwire angles
# =============================

plt.figure(figsize=(9, 6))

for target_key, runs in grouped_results.items():

    # Only select tripwire 0.94 mm cases
    if target_key[0] == "Trip" and target_key[1] == TARGET_TRIP_MM:

        angle = target_key[2]

        Re_list = []
        CD_list = []
        CD_error_list = []

        for run_data in sorted(runs, key=re_value):

            Re = re_value(run_data)

            # -----------------------------------
            # Option 1: if C_D batches already exist
            # -----------------------------------
            if "C_D_batches" in run_data:

                CD_mean, CD_error = mean_and_error_from_batches(
                    run_data["C_D_batches"]
                )

            # -----------------------------------
            # Option 2: if Cp batches exist
            # -----------------------------------
            elif "cp_batches" in run_data:

                CD_batches = []

                for cp_batch in run_data["cp_batches"]:
                    CD_batch = compute_cd(run_data["theta"], cp_batch)
                    CD_batches.append(CD_batch)

                CD_mean, CD_error = mean_and_error_from_batches(CD_batches)

            # -----------------------------------
            # Fallback: no batch data available
            # -----------------------------------
            else:
                CD_mean = compute_cd(run_data["theta"], run_data["cp"])
                CD_error = 0.0

                print(
                    f"Warning: No batch data found for {target_key}, {run_data['re']}. "
                    "Plotted without real error bar."
                )

            Re_list.append(Re)
            CD_list.append(CD_mean)
            CD_error_list.append(CD_error)

        plt.errorbar(
            Re_list,
            CD_list,
            yerr=CD_error_list,
            fmt="o-",
            capsize=4,
            label=f"A{angle}"
        )

plt.xlabel("Reynolds number")
plt.ylabel("$C_D$")
plt.title(f"$C_D$ vs Reynolds number with error bars - Trip wire {TARGET_TRIP_MM} mm")
plt.grid(True)
plt.legend(title="Trip angle")
plt.show()

## Third attempt

### Code

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# =============================
# User settings
# =============================

DATA_FOLDER = Path(r"C:\Users\mpciv\Downloads\LVM-20260319T152800Z-3-001\LVM")

LAYER1_COLS = (1, 30)      # pressure taps: layer 1
LAYER2_COLS = (33, 62)     # pressure taps: layer 2

BATCH_SIZE = 1000          # first 1000 samples, next 1000 samples, etc.


# =============================
# Small helper functions
# =============================

def read_lvm(file_path):
    return pd.read_csv(file_path, sep=r"\s+", header=None, engine="python")


def get_columns(df, start, end):
    """
    Extracts columns using 1-based inclusive indexing.
    Example: columns 1 to 30.
    """
    return df.iloc[:, start - 1:end].to_numpy()


def parse_filename(file_path):
    """
    Extracts information from filenames such as:

    Cyl100_Re250k_Clean.lvm
    Cyl100_Re250k_Trip_0.94_A27.lvm
    """

    parts = file_path.stem.split("_")

    re_label = [p for p in parts if p.startswith("Re")][0]

    if "Clean" in parts:
        return {
            "key": ("Clean",),
            "case": "Clean",
            "trip_mm": None,
            "angle": None,
            "re": re_label
        }

    if "Trip" in parts:
        trip_index = parts.index("Trip")
        trip_mm = parts[trip_index + 1]

        angle = [p.replace("A", "") for p in parts if p.startswith("A")][0]

        return {
            "key": ("Trip", trip_mm, angle),
            "case": "Trip",
            "trip_mm": trip_mm,
            "angle": angle,
            "re": re_label
        }

    return None


def re_to_number(re_label):
    """
    Converts Re250k to 250000.
    """
    return int(re_label.replace("Re", "").replace("k", "")) * 1000


def combine_layers(layer1_values, layer2_values):
    """
    Combines the two pressure tap layers into one ordered distribution.

    Layer 1: 0, 12, 24, ..., 348 degrees
    Layer 2: 6, 18, 30, ..., 354 degrees
    """

    theta1 = np.arange(0, 360, 12)
    theta2 = np.arange(6, 360, 12)

    theta = np.r_[theta1, theta2]
    values = np.r_[layer1_values, layer2_values]

    order = np.argsort(theta)

    return theta[order], values[order]


def compute_cd(theta_deg, cp):
    """
    Computes C_D by integrating Cp around the cylinder.
    """

    theta_closed = np.r_[theta_deg, 360.0]
    cp_closed = np.r_[cp, cp[0]]

    theta_rad = np.deg2rad(theta_closed)

    C_D = 0.5 * np.trapz(cp_closed * np.cos(theta_rad), theta_rad)

    return C_D


# =============================
# Main grouping code
# =============================

grouped_results = {}

for file_path in sorted(DATA_FOLDER.glob("*.lvm")):

    meta = parse_filename(file_path)

    if meta is None:
        print(f"Skipping file: {file_path.name}")
        continue

    df = read_lvm(file_path)

    # Extract pressure taps
    layer1 = get_columns(df, *LAYER1_COLS)
    layer2 = get_columns(df, *LAYER2_COLS)

    # Extract dynamic pressure.
    # This assumes pitot is column 63, directly after the pressure tap columns.
    q_inf = df.iloc[:, 62].to_numpy()

    # =============================
    # Full-signal mean Cp and C_D
    # =============================

    q_mean = np.mean(q_inf)

    layer1_mean = np.mean(layer1, axis=0)
    layer2_mean = np.mean(layer2, axis=0)

    cp_layer1 = layer1_mean / q_mean
    cp_layer2 = layer2_mean / q_mean

    theta, cp = combine_layers(cp_layer1, cp_layer2)

    C_D = compute_cd(theta, cp)

    # =============================
    # Batch C_D values
    # =============================

    C_D_batches = []

    n_samples = len(df)
    n_batches = n_samples // BATCH_SIZE

    for i in range(n_batches):

        start = i * BATCH_SIZE
        end = start + BATCH_SIZE

        layer1_batch = layer1[start:end, :]
        layer2_batch = layer2[start:end, :]
        q_batch = q_inf[start:end]

        q_batch_mean = np.mean(q_batch)

        layer1_batch_mean = np.mean(layer1_batch, axis=0)
        layer2_batch_mean = np.mean(layer2_batch, axis=0)

        cp_layer1_batch = layer1_batch_mean / q_batch_mean
        cp_layer2_batch = layer2_batch_mean / q_batch_mean

        theta_batch, cp_batch = combine_layers(cp_layer1_batch, cp_layer2_batch)

        C_D_batch = compute_cd(theta_batch, cp_batch)

        C_D_batches.append(C_D_batch)

    # =============================
    # Store results
    # =============================

    run_data = {
        "file": file_path.name,
        "re": meta["re"],
        "Re": re_to_number(meta["re"]),
        "case": meta["case"],
        "trip_mm": meta["trip_mm"],
        "angle": meta["angle"],
        "theta": theta,
        "cp": cp,
        "C_D": C_D,
        "C_D_batches": C_D_batches
    }

    key = meta["key"]

    if key not in grouped_results:
        grouped_results[key] = []

    grouped_results[key].append(run_data)


# =============================
# Print overview
# =============================

for key, runs in grouped_results.items():
    print(key, ":", len(runs), "runs")

### Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =============================
# Choose run
# =============================
target_key = ("Trip", "0.15", "69")
target_re = "Re100k"

# =============================
# Find the selected run
# =============================
runs = grouped_results[target_key]

run_data = None
for run in runs:
    if run["re"] == target_re:
        run_data = run
        break

if run_data is None:
    raise ValueError("Selected run was not found.")

# =============================
# Extract batch values
# =============================
cd_batches = np.asarray(run_data["C_D_batches"], dtype=float)

n_batches = len(cd_batches)
batch_numbers = np.arange(1, n_batches + 1)

cd_mean = np.mean(cd_batches)
cd_std = np.std(cd_batches, ddof=1)

# 95% confidence interval for 5 batches
t_value = 2.776
cd_error = t_value * cd_std / np.sqrt(n_batches)

# =============================
# Plot
# =============================
plt.figure(figsize=(8, 5))

plt.plot(
    batch_numbers,
    cd_batches,
    "o-",
    label="Batch $C_D$ values"
)

plt.axhline(
    cd_mean,
    color="black",
    linestyle="-",
    label=rf"Mean $C_D = {cd_mean:.3f}$"
)

plt.axhline(
    cd_mean + cd_error,
    color="gray",
    linestyle="--",
    label="95% confidence interval"
)

plt.axhline(
    cd_mean - cd_error,
    color="gray",
    linestyle="--"
)

plt.xlabel("Batch number")
plt.ylabel("$C_D$")
plt.title(f"Batch-based $C_D$ values for {target_key}, {target_re}")
plt.grid(True)
plt.legend()
plt.show()

print(f"C_D = {cd_mean:.4f} ± {cd_error:.4f}")



for i, cd in enumerate(cd_batches, start=1):
    print(f"Batch {i}: C_D = {cd:.4f}")

print(f"File: {FILE_PATH.name}")
print(f"Strouhal number St = {St:.4f}")

### All runs for one angle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

target_key = ("Trip", "0.94", "69")

def re_value(run):
    return int(run["re"].replace("Re", "").replace("k", ""))

for run_data in sorted(grouped_results[target_key], key=re_value):

    cd_batches = np.asarray(run_data["C_D_batches"], dtype=float)

    n_batches = len(cd_batches)
    batch_numbers = np.arange(1, n_batches + 1)

    cd_mean = np.mean(cd_batches)
    cd_std = np.std(cd_batches, ddof=1)

    t_value = 2.776
    cd_error = t_value * cd_std / np.sqrt(n_batches)

    plt.figure(figsize=(8, 5))

    plt.plot(batch_numbers, cd_batches, "o-", label="Batch $C_D$ values")

    plt.axhline(cd_mean, color="black", linestyle="-", label=rf"Mean $C_D = {cd_mean:.3f}$")
    plt.axhline(cd_mean + cd_error, color="gray", linestyle="--", label="95% confidence interval")
    plt.axhline(cd_mean - cd_error, color="gray", linestyle="--")

    plt.xlabel("Batch number")
    plt.ylabel("$C_D$")
    plt.title(f"Batch $C_D$ values for trip 0.94 mm at A{target_key[2]}, {run_data['re']}")
    plt.grid(True)
    plt.legend()
    plt.show()

    print(f"{run_data['re']}: C_D = {cd_mean:.4f} ± {cd_error:.4f}")
    print(f"Strouhal number St = {St:.4f}")

In [ ]:
5*10**5

# Day 1 velocities

In [ ]:
from sympy import * 


In [ ]:

T = 273.15 + 23.8 # K in the wind tunnel 17/3.
T_0 = 291.15  # reference temperature for Sutherland's law
S = 110.4  # Sutherland's constant for air in K
mu = 1.716*10**-5 * ((T/T_0)**1.5) * ((T_0+S) / (T + S))


Re_intervals = [50000, 100000, 150000, 200000, 250000]

for Re in Re_intervals:
    rho = 1.19736  # kg/m^3 FROM EXPERIMENTAL CONDITION PICTURE
    V = symbols("V")  # m/s
    D = 0.1  # m
    Re1 = Re
    Re2 = rho * V * D / mu

    eq = Eq(Re1, Re2)
    solution = solve(eq, V)
    print(f"Velocity for Re={Re1}: {solution[0]:.2f} m/s")

mu

# Day 2 velocities

In [4]:
from sympy import *
T = 273.15 + 25.9 # K in the wind tunnel 13/5.
T_0 = 291.15  # reference temperature for Sutherland's law
S = 110.4  # Sutherland's constant for air in K
mu = 1.716*10**-5 * ((T/T_0)**1.5) * ((T_0+S) / (T + S))

Re_intervals = [50000, 100000, 150000, 200000,170000, 250000,360000]

for Re in Re_intervals:
    rho = 1.16566  # kg/m^3 FROM EXPERIMENTAL CONDITION PICTURE
    V = symbols("V")  # m/s
    D = 0.1  # m
    Re1 = Re
    Re2 = rho * V * D / mu
    eq = Eq(Re1, Re2)
    solution = solve(eq, V)
    print(f"Velocity for Re={Re1}: {solution[0]:.2f} m/s")



Velocity for Re=50000: 7.51 m/s
Velocity for Re=100000: 15.03 m/s
Velocity for Re=150000: 22.54 m/s
Velocity for Re=200000: 30.06 m/s
Velocity for Re=170000: 25.55 m/s
Velocity for Re=250000: 37.57 m/s
Velocity for Re=360000: 54.10 m/s


In [ ]:
from sympy import *

In [ ]:
x = 4

f = ((0.4*10) + (0.6*x))

f

In [ ]:
(30.06 - 25.55)*3.6

In [ ]:
25.55 * 3.6

In [ ]:
5*10**5

In [ ]:
15.03*3.6